# Fibroblast ageing — publication figures

scRNA-seq + multiome + Xenium, drawn from the harmonised annotation produced in
`Fbs_smx_harmonize.ipynb` (2026-08-29):

| object | cells | annotation |
|---|---|---|
| `thyAgeing_Fb_scell_multiome_nuclei_anno_2026-08-20.zarr` | 85,289 | `taa_l3_cm` (scRNA-seq labels, kNN-transferred to the multiome nuclei) |
| `thyAgeing_Fb_scell_multiome_xenium_scvi_filt_2026-08-27.zarr` | 146,178 | `taa_l3_cmx` (the above, kNN-transferred to Xenium, + `Adipocyte`) |

Both annotations live in CSVs next to the objects, not in the objects themselves, so nothing here
depends on re-writing a multi-GB file.

**One vocabulary for the whole figure set.** `plotting.fb_pub_figs` holds the population colours,
the population display names, the modality colours and the age-group colours. Every panel takes
them from there, so a population is one colour and one name throughout. Change `POP_COLORS` /
`POP_NAMES` in §1 and re-run to recolour or rename everything at once.

---

## 1. Setup and the shared vocabulary

In [1]:
import os, sys, warnings
from datetime import datetime
warnings.filterwarnings('ignore')

import hdf5plugin                      # registers the blosc filter; must precede any h5 read
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt

REPO = '/nfs/team205/vk8/projects/thymus_ageing_atlas'
repo_path = f'{REPO}/FB_Vascular_compartment'
sys.path.insert(1, repo_path)
sys.path.insert(2, f'{REPO}/General_analysis/scripts')

import plotting.fb_pub_figs as P
import plotting.fb_factor_figs as G

today = datetime.today().strftime('%Y-%m-%d')
FIGS = f'{repo_path}/plots/Fb_ageing_publication'
RES  = f'{repo_path}/results/Fb_ageing_publication'
os.makedirs(FIGS, exist_ok=True); os.makedirs(RES, exist_ok=True)

P.set_style()
print(f'font   : {plt.rcParams["font.sans-serif"][0]} | base {plt.rcParams["font.size"]} pt '
      f'| ticks {plt.rcParams["xtick.labelsize"]} pt | pdf fonttype {plt.rcParams["pdf.fonttype"]}')
print(f'output : {FIGS}\n         {RES}')

font   : Arial | base 7.0 pt | ticks 5.0 pt | pdf fonttype 42
output : /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication
         /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/results/Fb_ageing_publication


### Colours and names — edit here, nowhere else

The **modality** colours are deliberately not the project blue/orange: those are `Fb-perilo` and
`Fb-interlo`, and a reader moving between the modality panel and the population panel would read
the same colour as two different things. The triple below maximises the minimum CIELAB distance to
every population and age colour (min ΔE 44.8) while staying far apart from each other (ΔE 67).
The near-black goes to the multiome arm because it is the smallest — a heavy colour on the largest
arm would swamp the panel.

**Age** uses the magenta tint ramp, matching the published atlas frequency figures.

In [2]:
P.POP_NAMES = {
    'Fb-interlo': 'Fb-interlo',
    'Fb-perilo':  'Fb-perilo',
    'Fb-med':     'Fb-med',
    'Fb-interm':  'Fb-interm',
    'Fb-TGFB+':   'Fb-TGFB+',
    'Fb-adipo':   'Fb-adipo',
    'Adipocyte':  'Adipocyte',
}
P.AGE_COLORS = P.AGE_COLORS_RAMP        # pink ramp everywhere, UMAPs and boxplots alike
P.sync_fb_factor_figs()                 # let fb_factor_figs (§7) use the same order and colours
POP_ORDER = P.POP_ORDER

pd.DataFrame({'level': POP_ORDER + P.MODALITY_ORDER + P.AGE_ORDER,
              'kind': ['population'] * len(POP_ORDER) + ['modality'] * 3 + ['age'] * 4,
              'name': [P.POP_NAMES[p] for p in POP_ORDER]
                      + [P.MODALITY_NAMES[m] for m in P.MODALITY_ORDER] + P.AGE_ORDER,
              'colour': [P.POP_COLORS[p] for p in POP_ORDER]
                        + [P.MODALITY_COLORS[m] for m in P.MODALITY_ORDER]
                        + [P.AGE_COLORS[a] for a in P.AGE_ORDER]})

,level,kind,name,colour
0,Fb-interlo,population,Fb-interlo,#FE6100
1,Fb-perilo,population,Fb-perilo,#648FFF
2,Fb-med,population,Fb-med,#DC267F
3,Fb-interm,population,Fb-interm,#18868F
4,Fb-TGFB+,population,Fb-TGFB+,#785EF0
5,Fb-adipo,population,Fb-adipo,#ffd473
6,Adipocyte,population,Adipocyte,#FFB000
7,scell,modality,scRNA-seq,#82CD29
8,multiome,modality,Multiome (snRNA),#252525
9,xenium,modality,Xenium,#8B0000


In [3]:
fig, axes = plt.subplots(1, 3, figsize=P.figsize_mm('one-half', 24))
for ax, (title, pal, order) in zip(axes, [
        ('populations', P.POP_COLORS, POP_ORDER),
        ('modality', P.MODALITY_COLORS, P.MODALITY_ORDER),
        ('age group', P.AGE_COLORS, P.AGE_ORDER)]):
    for i, k in enumerate(order):
        ax.barh(-i, 1, color=pal[k])
        ax.text(1.1, -i, k, va='center', fontsize=5)
    ax.set_xlim(0, 4); ax.set_axis_off(); ax.set_title(title)
plt.show()

## 2. UMAPs

Both objects are read with `h5py` alone: the obs columns the figures need plus `X_umap`, never `X`.
The joint object is 85,289 × 35,530 sparse with 170M non-zeros — `read_h5ad` costs ~1.4 GB for
`data` + `indices` before AnnData wraps anything, and none of this section needs it.

The harmonised labels and the refreshed donor metadata come from the two CSVs
`Fbs_smx_harmonize.ipynb` wrote, joined **by cell label**.

Points are drawn in a fixed random order so no modality or age group is systematically on top —
a panel where one arm is painted last always reads as if that arm dominates. Donors get their own
full-width figure: 40+ levels do not fit a quarter-width legend, and one that overflows is silently
cropped by `savefig.bbox='standard'`.

In [4]:
OBJ = f'{repo_path}/data/objects'
C2M_H5   = f'{OBJ}/thyAgeing_Fb_scell_multiome_nuclei_anno_2026-08-20.zarr'
CMX_H5   = f'{OBJ}/thyAgeing_Fb_scell_multiome_xenium_scvi_filt_2026-08-27.zarr'
C2M_ANNO = f'{OBJ}/thyAgeing_Fb_scell_multiome_anno_c2m_2026-08-29.csv'
CMX_ANNO = f'{OBJ}/thyAgeing_Fb_scell_multiome_xenium_anno_cmx_2026-08-29.csv'

def build_obs(h5, anno_csv, label_col):
    obs = P.read_obs(h5, ['batch', 'sample', 'donor', 'donor_comb'])
    anno = pd.read_csv(anno_csv, index_col=0)
    assert obs.index.is_unique and anno.index.is_unique
    assert len(obs.index.difference(anno.index)) == 0, 'cells missing from the annotation CSV'
    for col in ['donor', 'age', 'age_months', 'age_group', 'sex', label_col]:
        obs[col] = anno.loc[obs.index, col].values     # the CSV is authoritative for all of these
    obs['modality'] = obs['batch'].astype(str)
    obs[label_col] = pd.Categorical(obs[label_col].astype(str),
                                    categories=P.order_present(obs[label_col], POP_ORDER))
    obs['age_group'] = pd.Categorical(obs['age_group'].astype(str),
                                      categories=P.order_present(obs['age_group'], P.AGE_ORDER))
    return obs

obs_c2m, xy_c2m = build_obs(C2M_H5, C2M_ANNO, 'taa_l3_cm'), P.read_obsm(C2M_H5, 'X_umap')
obs_cmx, xy_cmx = build_obs(CMX_H5, CMX_ANNO, 'taa_l3_cmx'), P.read_obsm(CMX_H5, 'X_umap')

for name, obs, lab in [('taa_l3_cm  (scell+multiome)', obs_c2m, 'taa_l3_cm'),
                       ('taa_l3_cmx (+ xenium)', obs_cmx, 'taa_l3_cmx')]:
    print(f'--- {name}: {obs.shape[0]:,} cells')
    print(pd.crosstab(obs['modality'], obs[lab]).to_string(), '\n')

# donors from BOTH objects: the +Xenium object is a filtered subset, so a few donors exist only in
# the joint one and would otherwise fall through to grey
_don = pd.concat([obs_c2m[['donor', 'age_group']], obs_cmx[['donor', 'age_group']]]).astype(str)
donor_age = _don.drop_duplicates('donor').set_index('donor')['age_group'].to_dict()
donor_pal = P.donor_palette(sorted(donor_age))
print(f'{len(donor_pal)} donors: ' +
      str(pd.Series(donor_age).value_counts().reindex(P.AGE_ORDER).fillna(0).astype(int).to_dict()))

--- taa_l3_cm  (scell+multiome): 85,289 cells
taa_l3_cm  Fb-interlo  Fb-perilo  Fb-med  Fb-interm  Fb-TGFB+  Fb-adipo
modality                                                               
multiome         4743       5229    1263       2932       273       591
scell           18363      23063    7517      14815      1284      5216 

--- taa_l3_cmx (+ xenium): 146,178 cells
taa_l3_cmx  Fb-interlo  Fb-perilo  Fb-med  Fb-interm  Fb-TGFB+  Fb-adipo  Adipocyte
modality                                                                           
multiome          4631       5134    1044       2834       271       520        101
scell            17195      22629    7178      13998      1258      4847        430
xenium            9835      30115    8771       7340       184      1668       6195 

48 donors: {'infant': 22, 'paed': 7, 'adult': 15, 'aged': 4}


In [5]:
def umap_figs(obs, xy, label_col, tag, title):
    fig = P.umap_grid(
        obs, xy, ['modality', label_col, 'age_group'],
        titles=['Modality', 'Population', 'Age group'],
        palettes={'modality': P.MODALITY_COLORS, label_col: P.POP_COLORS,
                  'age_group': P.AGE_COLORS},
        orders={'modality': P.order_present(obs['modality'], P.MODALITY_ORDER),
                label_col: P.order_present(obs[label_col], POP_ORDER),
                'age_group': P.order_present(obs['age_group'], P.AGE_ORDER)},
        names={'modality': P.MODALITY_NAMES, label_col: P.POP_NAMES},
        ncols=3, point_size=0.35, width='double',
        panel_height_mm=52, legend_height_mm=18, suptitle=title)
    P.save(fig, f'{FIGS}/Fig1{tag}_umap_{tag}_{today}.pdf')
    plt.show()

    fig = P.umap_one(obs, xy, 'donor', title=f'Donor — {title}', palette=donor_pal,
                     order=sorted(obs['donor'].astype(str).unique()),
                     point_size=0.35, legend_ncol=5, width='double', panel_height_mm=58)
    P.save(fig, f'{FIGS}/Fig1{tag}2_umap_donor_{tag}_{today}.pdf')
    plt.show()

umap_figs(obs_c2m, xy_c2m, 'taa_l3_cm', 'a', 'scRNA-seq + multiome (taa_l3_cm)')

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig1a_umap_a_2026-10-28.{pdf}   182 x 76 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig1a2_umap_donor_a_2026-10-28.{pdf}   182 x 58 mm


In [6]:
umap_figs(obs_cmx, xy_cmx, 'taa_l3_cmx', 'b', 'scRNA-seq + multiome + Xenium (taa_l3_cmx)')

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig1b_umap_b_2026-10-28.{pdf}   182 x 76 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig1b2_umap_donor_b_2026-10-28.{pdf}   182 x 58 mm


## 3. Marker panels

Expression is pulled out of the objects by **streaming the CSR** — walking `indptr` in row blocks
and keeping only the wanted columns. Peak memory is one block, independent of file size; loading
the joint object to reach 60 genes would cost ~1.4 GB and has crashed a kernel before. `X` is
log-normalised in both objects (`layers/counts` holds the raw counts), so the streamed values are
what a dotplot wants.

`MEAN_ONLY_EXPRESSED` is set once here and passed to every dotplot. It changes what the **colour**
means: with `True` the colour is the mean over *expressing cells only*, so it reads as "how much,
where detected" and is independent of the dot area; with `False` it is the mean over all cells in
the group and therefore partly re-encodes the same detection rate the dot area already shows.

In [7]:
MEAN_ONLY_EXPRESSED = True          # colour = mean over expressing cells only

# VK's lists. The activity panel carries only PDGFRA/VIM as the pan-Fb reference; the full
# pan-Fb block stays on the identity panel, where the space is not competing with five programmes.
PAN_FB      = ['PDGFRA', 'VIM', 'DCN', 'PDPN', 'THY1']
PAN_FB_MIN  = ['PDGFRA', 'VIM']

IDENTITY_MARKERS = {
    'Fb-interlo': ['MFAP5', 'CD55', 'PI16', 'FBN1', 'PCOLCE2'],
    'Fb-perilo':  ['COL15A1', 'COLEC11', 'ALDH1A2', 'PTN', 'HAND2'],
    'Fb-med':     ['HLA-DRA', 'CD74', 'CCL19', 'CCL21', 'IL33'],
}
ACTIVITY_MARKERS = {
    'Fb-interm':  ['PTX3', 'FOSL1', 'TNFRSF12A', 'IL1RL1', 'HMGA1'],
    'Fibrotic':   ['TGFB1', 'SNAI1', 'SNAI2', 'COL1A1', 'COL3A1', 'COL5A1', 'SPARC'],
    'Activated':  ['ACTA2', 'THY1', 'LMCD1', 'FLNA', 'PRG4', 'ALDH1A3'],
    'Fb-adipo':   ['PPARG', 'PLIN1', 'LIPE', 'CIDEA', 'AQP7'],
}
# VK's revised adipose panel (2026-08-29): the progenitor block, then the adipocyte programme
# broken up by FUNCTION rather than by differentiation stage.
ADIPOSE_DIF = {
    'Adipocyte\nprogenitors': ['PDGFRA', 'PDGFRB', 'CD34', 'CD55', 'PI16', 'DPP4', 'IL33'],
    'lipid\nsynthesis':       ['FASN', 'ACLY', 'ACACA', 'DGAT1'],
    'fa transport':          ['CD36'],
    'lipases':               ['LPL', 'LIPE'],
    'tfs':                   ['CEBPA', 'PPARG'],
    'storage':               ['PLIN1', 'PLIN4'],
    'hormone':               ['LEP', 'ADIPOQ'],
}

ACTIVITY_PANEL = {'pan-Fb': PAN_FB_MIN, **ACTIVITY_MARKERS}
IDENTITY_PANEL = {'pan-Fb': PAN_FB, **IDENTITY_MARKERS}
ALL_PANEL      = {'pan-Fb': PAN_FB, **IDENTITY_MARKERS, **ACTIVITY_MARKERS}

GENES = sorted({g for d in [ALL_PANEL, ADIPOSE_DIF] for gs in d.values() for g in gs})
print(f'{len(GENES)} distinct genes across all panels')

55 distinct genes across all panels


In [8]:
E_c2m = P.stream_genes(C2M_H5, GENES)              # 85,289 x n_genes, log-normalised
A_c2m = P.marker_adata(E_c2m, obs_c2m)
A_c2m.obs['population'] = A_c2m.obs['taa_l3_cm']

E_cmx = P.stream_genes(CMX_H5, GENES)              # 146,178 cells, 4,807-gene Xenium panel
A_cmx = P.marker_adata(E_cmx, obs_cmx)
A_cmx.obs['population'] = A_cmx.obs['taa_l3_cmx']

# Several markers are simply not measurable on the 4,807-gene Xenium probe set. Filtering them out
# is not cosmetic: an off-panel gene draws as an all-zero row, which reads as 'not expressed'
# rather than 'not measured'.
print('genes dropped from the +Xenium panels (off the Xenium probe set):')
ALL_PANEL_X = P.filter_gene_dict(ALL_PANEL, A_cmx.var_names)
ADIPOSE_X   = P.filter_gene_dict(ADIPOSE_DIF, A_cmx.var_names)
ORDER_C2M = P.order_present(obs_c2m['taa_l3_cm'], POP_ORDER)
ORDER_CMX = P.order_present(obs_cmx['taa_l3_cmx'], POP_ORDER)
print('\nrow order, scell+multiome:', ORDER_C2M)
print('row order, +xenium        :', ORDER_CMX)


def dotplot_pair(adata, panel, order, stem, width_mm, height_mm):
    """Draw one panel twice -- gene-scaled and raw -- and save both.

    The two colour bars do NOT mean the same thing and the figures must not be mixed in one
    composite. Scaled: each gene min-max scaled across the populations, so the colour is
    'high or low FOR THIS GENE' and every row is legible whatever its absolute level.
    Unscaled: the raw log-normalised mean, so the colour is comparable BETWEEN genes and the
    panel is dominated by whatever is abundant -- VIM and DCN sit near the top of the scale
    and a specific but lowly expressed marker such as CIDEA or LEP is pale everywhere.
    """
    for scale, sfx in [('var', ''), (None, '_unscaled')]:
        P.dotplot(adata, panel, groupby='population', categories_order=order,
                  width_mm=width_mm, height_mm=height_mm, standard_scale=scale,
                  mean_only_expressed=MEAN_ONLY_EXPRESSED,
                  path=f'{FIGS}/{stem}{sfx}_{today}.pdf')

thyAgeing_Fb_scell_multiome_nuclei_anno_2026-08-20.zarr: 85,289 x 35,530 | 55/55 genes on file


thyAgeing_Fb_scell_multiome_xenium_scvi_filt_2026-08-27.zarr: 146,178 x 4,807 | 35/55 genes on file


  not in var_names: ['ACTA2', 'ALDH1A2', 'ALDH1A3', 'CCL21', 'CD74', 'CIDEA', 'COL15A1', 'COL1A1', 'COL3A1', 'COLEC11', 'DCN', 'FBN1', 'FLNA', 'HAND2', 'HLA-DRA', 'LMCD1', 'PCOLCE2', 'PTN', 'SPARC', 'VIM']


  rows 0/146,178


genes dropped from the +Xenium panels (off the Xenium probe set):
  pan-Fb: dropped ['VIM', 'DCN']
  Fb-interlo: dropped ['FBN1', 'PCOLCE2']
  Fb-perilo: dropped ['COL15A1', 'COLEC11', 'ALDH1A2', 'PTN', 'HAND2']
  Fb-med: dropped ['HLA-DRA', 'CD74', 'CCL21']
  Fibrotic: dropped ['COL1A1', 'COL3A1', 'SPARC']
  Activated: dropped ['ACTA2', 'LMCD1', 'FLNA', 'ALDH1A3']
  Fb-adipo: dropped ['CIDEA']

row order, scell+multiome: ['Fb-interlo', 'Fb-perilo', 'Fb-med', 'Fb-interm', 'Fb-TGFB+', 'Fb-adipo']
row order, +xenium        : ['Fb-interlo', 'Fb-perilo', 'Fb-med', 'Fb-interm', 'Fb-TGFB+', 'Fb-adipo', 'Adipocyte']


## 4. Marker dotplots — scRNA-seq + multiome

Rows are all populations of `taa_l3_cm`; columns are the marker programmes. Showing every
population rather than only the ones a panel is about is deliberate — a marker block only means
something against the states that do *not* carry it.

**Every panel is written twice**, by `dotplot_pair`: the default and a `_unscaled` companion.

- **scaled** (`standard_scale='var'`, the file without a suffix) — each gene min-max scaled across
  the populations. The colour reads as "high or low *for this gene*", so every row is legible
  whatever its absolute level. This is what makes a marker panel usable when the genes differ
  100-fold in expression.
- **unscaled** (`_unscaled`) — the raw log-normalised mean. The colour is now comparable *between*
  genes, at the cost that the panel is dominated by whatever is abundant: VIM and DCN sit near the
  top of the scale while a specific but lowly expressed marker such as CIDEA or LEP is pale in
  every population.

The two colour bars do not mean the same thing, so the two versions must not be mixed inside one
composite figure. The dot area is the raw fraction of cells expressing in both.

In [9]:
# --- MAIN: activity states
dotplot_pair(A_c2m, ACTIVITY_PANEL, ORDER_C2M,
             'Fig2a_dotplot_activity_scellMultiome', width_mm=140, height_mm=32)

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig2a_dotplot_activity_scellMultiome_2026-10-28.pdf   140 x 32 mm (cropped to content)


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig2a_dotplot_activity_scellMultiome_unscaled_2026-10-28.pdf   140 x 32 mm (cropped to content)


In [10]:
# --- SUP: identity states
dotplot_pair(A_c2m, IDENTITY_PANEL, ORDER_C2M,
             'Fig2b_dotplot_identity_scellMultiome', width_mm=120, height_mm=32)

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig2b_dotplot_identity_scellMultiome_2026-10-28.pdf   120 x 32 mm (cropped to content)


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig2b_dotplot_identity_scellMultiome_unscaled_2026-10-28.pdf   120 x 32 mm (cropped to content)


## 5. Marker dotplots — scRNA-seq + multiome + Xenium

`taa_l3_cmx` adds `Adipocyte`, a state the dissociated data does not see at all: mature adipocytes
are lost in dissociation, so the adipose end of the axis exists only in situ. That is why the
staged adipogenesis panel is drawn on this object rather than the joint one.

In [11]:
# --- MAIN: staged adipose differentiation
dotplot_pair(A_cmx, ADIPOSE_X, ORDER_CMX,
             'Fig3a_dotplot_adiposeDifferentiation_cmx', width_mm=140, height_mm=34)

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig3a_dotplot_adiposeDifferentiation_cmx_2026-10-28.pdf   140 x 34 mm (cropped to content)


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig3a_dotplot_adiposeDifferentiation_cmx_unscaled_2026-10-28.pdf   140 x 34 mm (cropped to content)


In [12]:
# --- SUP: identity + activity together
dotplot_pair(A_cmx, ALL_PANEL_X, ORDER_CMX,
             'Fig3b_dotplot_identityActivity_cmx', width_mm=183, height_mm=34)

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig3b_dotplot_identityActivity_cmx_2026-10-28.pdf   183 x 34 mm (cropped to content)


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig3b_dotplot_identityActivity_cmx_unscaled_2026-10-28.pdf   183 x 34 mm (cropped to content)


### 5.1 The adipose end of the axis, one row per modality

The same staged panel restricted to the two adipose states and split by assay, so the scRNA-seq and
Xenium reads of each state sit next to each other. Multiome is left out — VK asked for scRNA-seq and
Xenium, and the multiome arm contributes 520 `Fb-adipo` and 101 `Adipocyte` nuclei, too few to add a
useful pair of rows.

Both arms are drawn from the **same object and therefore the same 4,807-gene space**, so a
difference between two rows is a difference in the cells, not in the gene set. It is still not a
clean assay comparison: Xenium measures a targeted panel in situ with no dissociation loss, and the
`Adipocyte (scRNA-seq)` row is 430 cells that survived dissociation, which is exactly the
population the protocol is worst at recovering.

In [13]:
ADIPO_STATES = ['Fb-adipo', 'Adipocyte']
ARMS = {'scell': 'scRNA-seq', 'xenium': 'Xenium'}

sel = (A_cmx.obs['taa_l3_cmx'].astype(str).isin(ADIPO_STATES)
       & A_cmx.obs['batch'].astype(str).isin(ARMS)).values
A_ad = A_cmx[sel].copy()
# `dotplot_pair` groups by 'population', which A_cmx already carries as the plain state -- the
# state x assay label has to REPLACE it, not sit beside it, or the dotplot groups by state while
# `categories_order` names the combined levels and scanpy rejects the mismatch.
A_ad.obs['population'] = [f'{p} ({ARMS[b]})' for p, b in
                          zip(A_ad.obs['taa_l3_cmx'].astype(str),
                              A_ad.obs['batch'].astype(str))]
# the order VK asked for: each state's two assays adjacent, Fb-adipo before Adipocyte
ORDER_AD = [f'{p} ({ARMS[b]})' for p in ADIPO_STATES for b in ARMS]
ORDER_AD = [g for g in ORDER_AD if g in set(A_ad.obs['population'])]
print(A_ad.obs['population'].value_counts().reindex(ORDER_AD).to_string())

population
Fb-adipo (scRNA-seq)     4847
Fb-adipo (Xenium)        1668
Adipocyte (scRNA-seq)     430
Adipocyte (Xenium)       6195


In [14]:
# 4 rows would fit in ~22 mm, but scanpy stacks the size legend and the colour bar in a fixed
# column beside the grid and does not shrink them with the panel -- below ~30 mm the two collide
dotplot_pair(A_ad, ADIPOSE_X, ORDER_AD,
             'Fig3c_dotplot_adiposeDifferentiation_byModality',
             width_mm=140, height_mm=30)

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig3c_dotplot_adiposeDifferentiation_byModality_2026-10-28.pdf   140 x 30 mm (cropped to content)


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig3c_dotplot_adiposeDifferentiation_byModality_unscaled_2026-10-28.pdf   140 x 30 mm (cropped to content)


### 5.2 The same, with the dissociated arm pooled

`Fig3c` again, with `Fb-adipo (scRNA-seq)` and `Adipocyte (scRNA-seq)` merged into one row, so the
dissociated arm contributes a single adipose group against the two in-situ ones.

The merge is defensible on this object rather than merely convenient: `Adipocyte` at `taa_l3_cmx`
is a Xenium-led call (6,195 of its 6,726 cells are Xenium), and the 430 cells it picks up on the
scRNA-seq side come from a leiden cluster defined on the joint embedding — 312 of them were
`Fb-adipo` at `taa_l3_cm`, so the split within the dissociated arm is largely a projection of the
in-situ structure rather than something the dissociated data resolves on its own. Pooling states
what that arm can actually support: one adipose population, of 5,277 cells.

Read the two figures together, not one instead of the other — `Fig3c` keeps the split visible so
the reader can judge it, and this one is the conservative version.

In [15]:
POOLED = 'Fb-adipo + Adipocyte (scRNA-seq)'
A_ad2 = A_ad.copy()
A_ad2.obs['population'] = (A_ad2.obs['population'].astype(str)
                           .replace({f'{p} (scRNA-seq)': POOLED for p in ADIPO_STATES}))
ORDER_AD2 = [POOLED] + [f'{p} (Xenium)' for p in ADIPO_STATES]
ORDER_AD2 = [g for g in ORDER_AD2 if g in set(A_ad2.obs['population'])]
print(A_ad2.obs['population'].value_counts().reindex(ORDER_AD2).to_string())

population
Fb-adipo + Adipocyte (scRNA-seq)    5277
Fb-adipo (Xenium)                   1668
Adipocyte (Xenium)                  6195


In [16]:
dotplot_pair(A_ad2, ADIPOSE_X, ORDER_AD2,
             'Fig3c2_dotplot_adiposeDifferentiation_byModality_scellPooled',
             width_mm=140, height_mm=30)

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig3c2_dotplot_adiposeDifferentiation_byModality_scellPooled_2026-10-28.pdf   140 x 30 mm (cropped to content)


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig3c2_dotplot_adiposeDifferentiation_byModality_scellPooled_unscaled_2026-10-28.pdf   140 x 30 mm (cropped to content)


## 6. Abundance across age — scRNA-seq atlas

The dissociated atlas (`thyAgeing_all_scvi_v6_2026-05-01`, obs only) with the fibroblast labels
replaced by the harmonised `taa_l3_cm` ones. `aged` is excluded throughout.

- **Fig 4a** — stacked bars: composition of the **fibroblast** compartment across infant / paed /
  adult.
- **Fig 4b, 4c** — boxplots: per-donor frequency of each Fb and vascular state **within the
  Fb + vascular compartment**, under two sort brackets — `TOT + CD45−` and `CD45−` alone.

**Two filters, both applied to every panel:**

1. a **sample** enters only if it has ≥ `MIN_SAMPLE_CELLS` Fb + vascular cells;
2. a **population** is plotted only if it has ≥ `MIN_POP_CELLS` cells across those samples **and**
   reaches ≥ `MIN_POP_FREQ` of the compartment as a median in at least one age group.

The second half of that rule is what actually does the work, and the cell count alone would not
have: `Fb-TGFB+` has thousands of cells in total yet a median of 0.02–0.16 % of the compartment in
every age group, so its boxes sit on the zero line and the Kruskal–Wallis is testing sampling
noise. The count filter guards against a population that is barely observed; the frequency filter
against one that is observed but never abundant. Both thresholds are printed with the full table
below, so raising `MIN_POP_FREQ` to 0.02 (which would also drop `EC-lymphatic` and `EC-cap_lipid`,
both at ~1.2 %) is a one-line change. Populations dropped here are still shown in the UMAPs, the
dotplots and the stacked bars — this filter applies to the frequency test only.

**The denominator has to be within-compartment**, and that is forced by the design rather than
chosen: CD45− depletes immune cells, so "fraction of all cells" is not comparable between the two
brackets. Statistics are Kruskal–Wallis across the three age groups per cell type, with a post-hoc
Dunn test (FDR-BH **within** a cell type) where KW p < 0.05. **KW p-values are not corrected across
cell types** — the convention the atlas frequency figures use, and one that belongs in the legend
rather than left implicit.

In [17]:
from anndata_compat import read_h5ad_light
from anno_levels import get_ct_levels
from utils import freq_by_donor
from plotting.utils import calc_figsize, plot_grouped_boxplot
from plotting.pop_stacked import population_stacked_plot, samples_passing, compartment_palette

ATLAS = f'{REPO}/General_analysis/data/objects/rna/thyAgeing_all_scvi_v6_2026-05-01.zarr'
CURATED = (f'{REPO}/General_analysis/data/objects/rna/'
           'thyAgeing_all_scvi_v5_2025-10-15_curatedAnno_v11.csv')

obs_at = read_h5ad_light(ATLAS, X=False, obsm=(), verbose=False).obs.copy()

# v6 carries no taa_l* columns -- the cell types live in the barcode-keyed curated annotation
anno_cols = ['taa_l0', 'taa_l1', 'taa_l2', 'taa_l2_v2', 'taa_l3', 'taa_l3_v2', 'taa_l4', 'taa_l5']
curated = pd.read_csv(CURATED, index_col=0)
obs_at = obs_at.drop(columns=[c for c in anno_cols if c in obs_at.columns]).join(curated[anno_cols])

# annotated cells only -- the same filter General_analysis/01a_freqAnalysis_freq applies
keep = (~obs_at['taa_l5'].astype(str).str.contains('locnt|-sp|explore', na=True)
        & obs_at['taa_l5'].notna())
print(f'{obs_at.shape[0]:,} cells; annotated and kept: {int(keep.sum()):,} ({keep.mean():.1%})')
obs_at = obs_at.loc[keep.values].copy()

# harmonised fibroblast labels
anno_c2m = pd.read_csv(C2M_ANNO, index_col=0)
new_lab = anno_c2m.loc[anno_c2m.index.intersection(obs_at.index), 'taa_l3_cm']
is_fb = obs_at['taa_l1'].astype(str) == 'Fb'
obs_at['taa_l3_upd'] = obs_at['taa_l3'].astype(str)
obs_at.loc[new_lab.index, 'taa_l3_upd'] = new_lab.values
# .astype(str) turns a missing label into the STRING 'nan', which population_stacked_plot would
# keep as a cell type instead of dropping it -- put the real NaN back
obs_at['taa_l3_upd'] = obs_at['taa_l3_upd'].replace({'nan': np.nan, 'None': np.nan, '': np.nan})
print(f'Fb cells {int(is_fb.sum()):,}; covered by taa_l3_cm '
      f'{int(is_fb.loc[new_lab.index].sum()):,}; non-Fb relabelled '
      f'{len(new_lab) - int(is_fb.loc[new_lab.index].sum())} (should be 0)')
pd.crosstab(obs_at.loc[is_fb, 'taa_l3'].astype(str), obs_at.loc[is_fb, 'taa_l3_upd'].astype(str))

1,028,237 cells; annotated and kept: 1,021,066 (99.3%)


Fb cells 70,258; covered by taa_l3_cm 70,258; non-Fb relabelled 0 (should be 0)


taa_l3_upd,Fb-TGFB+,Fb-adipo,Fb-interlo,Fb-interm,Fb-med,Fb-perilo
taa_l3,,,,,,
Fb-adipo,4,5216,0,0,0,0
Fb-interlo,1048,0,18363,0,0,0
Fb-interm,221,0,0,14815,0,0
Fb-med,6,0,0,0,7517,0
Fb-perilo,5,0,0,0,0,23063


In [18]:
FB_VASC = ['Fb', 'EC', 'Mural']
AGE3 = P.AGE_ORDER_NOAGED                 # ['infant', 'paed', 'adult'] -- 'aged' excluded
MIN_SAMPLE_CELLS = 200                    # Fb + vascular cells a sample must have
MIN_POP_CELLS    = 200                    # cells a population must have across those samples
MIN_POP_FREQ     = 0.01                   # and it must reach 1% of the compartment in some age group
BRACKETS = {'TOTCD45N': ('TOT', 'CD45N', 'CD45NCD3N'),
            'CD45N':    ('CD45N', 'CD45NCD3N')}

sub = obs_at.loc[obs_at['taa_l1'].astype(str).isin(FB_VASC),
                 ['taa_l1', 'taa_l3_upd']].astype(str).drop_duplicates()
ct2comp = dict(zip(sub['taa_l3_upd'], sub['taa_l1']))

def ordered_cts(comps):
    out = []
    for comp in comps:
        here = [ct for ct in ct2comp if ct2comp[ct] == comp]
        if comp == 'Fb':
            out += [p for p in POP_ORDER if p in here] + sorted(set(here) - set(POP_ORDER))
        else:
            curated_order = [ct for ct in get_ct_levels('taa_l3', taa_l1=comp) if ct in here]
            out += curated_order + sorted(set(here) - set(curated_order))
    return out

CT_FB, CT_FBVASC = ordered_cts(['Fb']), ordered_cts(FB_VASC)
print('Fb states     :', CT_FB)
print('Fb + vascular :', CT_FBVASC)

# Fb states keep the shared POP_COLORS; vascular states get tints of their compartment's base
# colour. One dict feeds the bars and the boxplots.
vasc = [ct for ct in CT_FBVASC if ct2comp[ct] != 'Fb']
PALETTE = {**compartment_palette(vasc, ct2comp={ct: ct2comp[ct] for ct in vasc}),
           **{ct: P.POP_COLORS[ct] for ct in CT_FB if ct in P.POP_COLORS}}
for ct in CT_FBVASC:
    PALETTE.setdefault(ct, '#cccccc')

Latest file: thyAgeing_full_curatedAnno_v12_2025-09-02_levels.xlsx


Latest file: thyAgeing_full_curatedAnno_v12_2025-09-02_levels.xlsx
Fb states     : ['Fb-interlo', 'Fb-perilo', 'Fb-med', 'Fb-interm', 'Fb-TGFB+', 'Fb-adipo']
Fb + vascular : ['Fb-interlo', 'Fb-perilo', 'Fb-med', 'Fb-interm', 'Fb-TGFB+', 'Fb-adipo', 'EC-art', 'EC-cap', 'EC-cap_lipid', 'EC-ven', 'EC-peri', 'EC-lymphatic', 'SMC', 'Pericyte']


In [19]:
# --- Fig 4a: composition of the fibroblast compartment across age
fig, tot_fb, sub_fb = population_stacked_plot(
    obs_at, col_cell_type='taa_l3_upd', ctypes=CT_FB, palette=PALETTE,
    age_order=AGE3, comp_col='taa_l1', comp_values=['Fb'],
    comp_sorts=BRACKETS['TOTCD45N'], tot_sorts=('TOT',), min_cells=MIN_SAMPLE_CELLS,
    figsize=calc_figsize(width=42, height=48),
    plt_path=f'{FIGS}/Fig4a_stackedbar_Fb_{today}.pdf', return_data=True)
plt.show()
sub_fb.to_csv(f'{RES}/Fig4a_stackedbar_Fb_{today}.csv', index=False)
tot_fb

top panel: 64 samples / 38 donors from sorts ['TOT']


bottom panel: 27 samples / 19 donors passing >= 200 taa_l1 cells in sorts ['TOT', 'CD45N', 'CD45NCD3N']
  donors per age group: infant=6, paed=2, adult=11


,age_group,mean_prop,sd_prop,n_donors
0,infant,0.000550,0.001207,19
1,paed,0.001087,0.000768,6
2,adult,0.043817,0.049790,13


In [20]:
# Per-donor frequencies within Fb + vascular, one table per sort bracket.
# Both filters are applied and reported here rather than inside the plotting call, so the panel and
# the numbers below it can never disagree about which samples and populations went in.
age3 = obs_at['age_group'].astype(str).isin(AGE3)
in_fbvasc = obs_at['taa_l1'].astype(str).isin(FB_VASC)

freq, kept_pops = {}, {}
for tag, sorts in BRACKETS.items():
    in_bracket = obs_at['sort'].astype(str).isin(sorts)
    keep_samples = samples_passing(obs_at.loc[age3 & in_bracket], comp_col='taa_l1',
                                   comp_values=FB_VASC, sample_col='sample',
                                   min_cells=MIN_SAMPLE_CELLS)
    sel = obs_at.loc[age3 & in_bracket & in_fbvasc & obs_at['sample'].isin(keep_samples)].copy()
    for col in ['sample', 'donor', 'taa_l3_upd']:
        if isinstance(sel[col].dtype, pd.CategoricalDtype):
            sel[col] = sel[col].cat.remove_unused_categories()

    df = freq_by_donor(sel, sample_col='sample', donor_col='donor',
                       summary_col='taa_l3_upd', add_meta=['age_group'])
    df = df.loc[df['age_group'].astype(str).isin(AGE3)]

    # Two criteria, because a pooled cell count does NOT catch the populations that look like
    # noise in a frequency boxplot: Fb-TGFB+ has thousands of cells in total yet a median of
    # 0.02-0.16% of the compartment in every age group, so its boxes sit on the zero line and the
    # Kruskal-Wallis is testing sampling noise. The count filter guards against a population that
    # is barely observed; the frequency filter against one that is observed but never abundant.
    counts = sel['taa_l3_upd'].astype(str).value_counts()
    med = (df.pivot_table(index='taa_l3_upd', columns='age_group', values='mean_prop',
                          aggfunc='median', observed=False).max(axis=1))
    verdict = pd.DataFrame({'cells': [int(counts.get(ct, 0)) for ct in CT_FBVASC],
                            'max_median_freq': [float(med.get(ct, 0)) for ct in CT_FBVASC]},
                           index=CT_FBVASC)
    verdict['keep'] = ((verdict['cells'] >= MIN_POP_CELLS)
                       & (verdict['max_median_freq'] >= MIN_POP_FREQ))
    kept_pops[tag] = verdict.index[verdict['keep']].tolist()
    df = df.loc[df['taa_l3_upd'].astype(str).isin(kept_pops[tag])].copy()
    # statannotations reads the hue levels from the CATEGORICAL DTYPE, not from the rows: leaving
    # 'aged' in the dtype makes it look up a level that is not in hue_order and raise
    df['age_group'] = pd.Categorical(df['age_group'].astype(str), categories=AGE3, ordered=True)
    df['taa_l3_upd'] = df['taa_l3_upd'].astype(str)
    freq[tag] = df
    df.to_csv(f'{RES}/Fig4_freq_{tag}_byDonor_{today}.csv', index=False)

    donors = sel.groupby(sel['age_group'].astype(str))['donor'].nunique().reindex(AGE3).fillna(0)
    print(f'--- {tag}  sorts {sorts}')
    print(f'    {sel["sample"].nunique()} samples / {sel["donor"].nunique()} donors '
          f'(infant {int(donors["infant"])}, paed {int(donors["paed"])}, '
          f'adult {int(donors["adult"])}); {len(sel):,} Fb+vascular cells')
    print(verdict.assign(max_median_freq=verdict['max_median_freq'].map('{:.4f}'.format))
          .to_string())
    print(f'    kept {len(kept_pops[tag])}/{len(CT_FBVASC)}; dropped '
          f'{verdict.index[~verdict["keep"]].tolist()}\n')

--- TOTCD45N  sorts ('TOT', 'CD45N', 'CD45NCD3N')
    31 samples / 22 donors (infant 8, paed 3, adult 11); 121,256 Fb+vascular cells
              cells max_median_freq   keep
Fb-interlo    14379          0.0997   True
Fb-perilo     20046          0.3271   True
Fb-med         6693          0.0959   True
Fb-interm     13988          0.0368   True
Fb-TGFB+       1015          0.0016  False
Fb-adipo       4003          0.0265   True
EC-art         4395          0.0643   True
EC-cap        12415          0.0856   True
EC-cap_lipid   1830          0.0125   True
EC-ven        18917          0.1891   True
EC-peri         963          0.0010  False
EC-lymphatic   1040          0.0115   True
SMC           15770          0.2181   True
Pericyte       5802          0.0712   True
    kept 12/14; dropped ['Fb-TGFB+', 'EC-peri']



--- CD45N  sorts ('CD45N', 'CD45NCD3N')
    24 samples / 21 donors (infant 8, paed 3, adult 10); 112,889 Fb+vascular cells
              cells max_median_freq   keep
Fb-interlo    13512          0.0997   True
Fb-perilo     18301          0.3271   True
Fb-med         6409          0.0959   True
Fb-interm     13138          0.0428   True
Fb-TGFB+        918          0.0016  False
Fb-adipo       3639          0.0249   True
EC-art         4072          0.0643   True
EC-cap        11798          0.0856   True
EC-cap_lipid   1469          0.0125   True
EC-ven        17735          0.1891   True
EC-peri         759          0.0010  False
EC-lymphatic    949          0.0115   True
SMC           14748          0.2181   True
Pericyte       5442          0.0712   True
    kept 12/14; dropped ['Fb-TGFB+', 'EC-peri']



In [21]:
for tag in BRACKETS:
    for suffix in ['_kruskal.csv', '_dunn.csv', '_summaryStats.csv']:
        f = f'{RES}/Fig4_freq_{tag}{suffix}'        # stale stats would be read back below
        if os.path.exists(f):
            os.remove(f)

panel = {'TOTCD45N': ('4b', 'TOT + CD45\u2212 samples'), 'CD45N': ('4c', 'CD45\u2212 samples')}
for tag, (fignum, label) in panel.items():
    df = freq[tag]
    order = [ct for ct in kept_pops[tag] if ct in set(df['taa_l3_upd'])]
    plot_grouped_boxplot(data=df, x='taa_l3_upd', y='mean_prop', hue='age_group',
                         order=order, hue_order=AGE3,
                         x_label='Cell population', y_label='Frequency (within Fb + vascular)',
                         legend_title='Age group', add_stats=True, format_percent=True,
                         palette=[P.AGE_COLORS[a] for a in AGE3],
                         figsize=(0.42 * len(order) + 1.6, 3.2), remove_legend=False,
                         legend_kwargs={'bbox_to_anchor': (1.01, 1), 'loc': 'upper left',
                                        'frameon': False},
                         save_stats=f'{RES}/Fig4_freq_{tag}')
    plt.gca().set_title(label, fontsize=plt.rcParams['axes.titlesize'])
    plt.gca().yaxis.set_major_locator(plt.MaxNLocator(4))
    plt.savefig(f'{FIGS}/Fig{fignum}_boxplot_freq_{tag}_{today}.pdf', bbox_inches='tight')
    plt.show()

Performing Kruskal-Wallis and Dunn tests


Performing Kruskal-Wallis and Dunn tests


In [22]:
# Median frequency per age group + the Kruskal-Wallis result, read back from the files the
# figures just wrote so the table and the panels cannot drift apart.
summary = []
for tag in BRACKETS:
    kwf = f'{RES}/Fig4_freq_{tag}_kruskal.csv'
    if not os.path.exists(kwf):
        continue
    kw = pd.read_csv(kwf, index_col=0)
    kw.index = kw.index.astype(str)
    med = (freq[tag].pivot_table(index='taa_l3_upd', columns='age_group', values='mean_prop',
                                 aggfunc='median', observed=False)
           .reindex([ct for ct in kept_pops[tag]]))
    med.columns = [str(x) for x in med.columns]     # a CategoricalIndex takes no new columns
    med.index = med.index.astype(str)
    t = med.join(kw[[x for x in ['statistic', 'pvalue', 'pvalue_signif'] if x in kw.columns]])
    t.insert(0, 'bracket', tag)
    t['adult_vs_infant'] = t['adult'] / t['infant']
    summary.append(t.rename_axis('cell_type'))
summary = pd.concat(summary).reset_index()
summary.to_csv(f'{RES}/Fig4_freq_KWsummary_{today}.csv', index=False)
summary.round(4)

,cell_type,bracket,infant,paed,adult,statistic,pvalue,pvalue_signif,adult_vs_infant
0,Fb-interlo,TOTCD45N,0.0456,0.0997,0.0783,5.0905,0.0785,NaN,1.7188
1,Fb-perilo,TOTCD45N,0.2388,0.3271,0.1571,4.0144,0.1344,NaN,0.6577
2,Fb-med,TOTCD45N,0.0959,0.0848,0.0257,8.5430,0.0140,*,0.2674
3,Fb-interm,TOTCD45N,0.0016,0.0124,0.0368,11.6441,0.0030,**,22.7745
4,Fb-adipo,TOTCD45N,0.0030,0.0093,0.0265,8.1118,0.0173,*,8.8629
5,EC-art,TOTCD45N,0.0643,0.0159,0.0207,5.9087,0.0521,NaN,0.3215
6,EC-cap,TOTCD45N,0.0856,0.0467,0.0547,2.3964,0.3017,NaN,0.6388
7,EC-cap_lipid,TOTCD45N,0.0125,0.0000,0.0107,7.4073,0.0246,*,0.8611
8,EC-ven,TOTCD45N,0.1891,0.0280,0.1196,5.1010,0.0780,NaN,0.6323
9,EC-lymphatic,TOTCD45N,0.0083,0.0115,0.0065,0.4251,0.8085,NaN,0.7854


### 6.4 Adipose differentiation across the whole stroma

The staged adipogenesis panel against every stromal population at `taa_l3`, not just the
fibroblasts — TEC, Fb, EC and Mural — to show which of these genes are actually fibroblast-restricted
and which are shared with the epithelium or the vasculature. Population order comes from the curated
levels table via `get_ct_levels`, with the fibroblast block replaced by the harmonised `taa_l3_upd`
states so the row labels match every other figure here.

It lives in this section rather than with the other adipose dotplots because it needs the atlas
object, which §6 has already loaded.

**The atlas `X` is raw `uint32` counts and the object has no normalised layer**, unlike the two Fb
objects whose `X` is already log-normalised. It is streamed with `lognorm=True`, which accumulates
the full row sum over all 35,530 genes in the same pass and returns `log1p(x / libsize * 1e4)` —
the same quantity `sc.pp.normalize_total` + `sc.pp.log1p` would give. Streaming 1.57 billion
non-zeros takes a few minutes, so the result is cached; delete the parquet to recompute.

In [23]:
ADIPO_CACHE = f'{RES}/atlas_adiposeGenes_lognorm.parquet'
GENES_AD = [g for gs in ADIPOSE_DIF.values() for g in gs]

if os.path.exists(ADIPO_CACHE):
    E_at = pd.read_parquet(ADIPO_CACHE)
    print(f'cached  {ADIPO_CACHE}  {E_at.shape}')
else:
    E_at, ln = P.stream_genes(ATLAS, GENES_AD, lognorm=True, return_libsize=True)
    # the streamed row sums must reproduce the library size the object recorded at QC time;
    # if they do not, X is not what this cell assumes it is
    chk = obs_at.index.intersection(ln.index)
    r = np.corrcoef(ln.loc[chk], obs_at.loc[chk, 'n_counts'].astype(float))[0, 1]
    print(f'streamed libsize vs obs n_counts: r = {r:.4f}')
    assert r > 0.99, 'streamed row sums disagree with n_counts -- X may not be raw counts'
    E_at.to_parquet(ADIPO_CACHE)
    print(f'-> {ADIPO_CACHE}')

cached  /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/results/Fb_ageing_publication/atlas_adiposeGenes_lognorm.parquet  (1028237, 20)


In [24]:
STROMA = ['TEC', 'Fb', 'EC', 'Mural', 'Adipocyte']
present_l1 = [c for c in STROMA if (obs_at['taa_l1'].astype(str) == c).any()]
print('stromal compartments in the atlas:', present_l1,
      '| absent:', [c for c in STROMA if c not in present_l1])

st = obs_at.loc[obs_at['taa_l1'].astype(str).isin(present_l1)].copy()
st = st.loc[st['taa_l3_upd'].notna()]
ct2comp_st = dict(zip(st['taa_l3_upd'].astype(str), st['taa_l1'].astype(str)))

ORDER_ST = []
for comp in present_l1:
    here = [ct for ct in ct2comp_st if ct2comp_st[ct] == comp]
    if comp == 'Fb':
        ORDER_ST += [p for p in POP_ORDER if p in here] + sorted(set(here) - set(POP_ORDER))
    else:
        curated = [ct for ct in get_ct_levels('taa_l3', taa_l1=comp) if ct in here]
        ORDER_ST += curated + sorted(set(here) - set(curated))

idx = st.index.intersection(E_at.index)
A_st = P.marker_adata(E_at.loc[idx], st.loc[idx])
A_st.obs['population'] = A_st.obs['taa_l3_upd'].astype(str)
ADIPOSE_ST = P.filter_gene_dict(ADIPOSE_DIF, A_st.var_names)
print(f'\n{A_st.n_obs:,} stromal cells, {len(ORDER_ST)} populations')
print(A_st.obs['population'].value_counts().reindex(ORDER_ST).to_string())

stromal compartments in the atlas: ['TEC', 'Fb', 'EC', 'Mural'] | absent: ['Adipocyte']


Latest file: thyAgeing_full_curatedAnno_v12_2025-09-02_levels.xlsx
Latest file: thyAgeing_full_curatedAnno_v12_2025-09-02_levels.xlsx
Latest file: thyAgeing_full_curatedAnno_v12_2025-09-02_levels.xlsx



236,206 stromal cells, 21 populations
population
cTEC            15291
mcTEC           29430
mTECI           24493
mTECII           3216
mTECIII          4506
TEC-mim          8255
TEC-EMT          4519
Fb-interlo      18363
Fb-perilo       23063
Fb-med           7517
Fb-interm       14815
Fb-TGFB+         1284
Fb-adipo         5216
EC-art           5569
EC-cap          14993
EC-cap_lipid     2761
EC-ven          23861
EC-peri          1546
EC-lymphatic     1675
SMC             18617
Pericyte         7216


In [25]:
dotplot_pair(A_st, ADIPOSE_ST, ORDER_ST,
             'Fig3d_dotplot_adiposeDifferentiation_stroma_taa_l3',
             width_mm=140, height_mm=max(28.0, 3.4 * len(ORDER_ST)))

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig3d_dotplot_adiposeDifferentiation_stroma_taa_l3_2026-10-28.pdf   140 x 71 mm (cropped to content)


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig3d_dotplot_adiposeDifferentiation_stroma_taa_l3_unscaled_2026-10-28.pdf   140 x 71 mm (cropped to content)


## 7. iNMF factor usage across the new annotation

The k=16, λ=5 pyliger iNMF model from `Fb_factors_05/06/07`, re-cut against `taa_l3_cm` instead of
the `taa_l4` / `celltype_v3_L3` pair the earlier figures used. Factors in rows, populations in
columns with the annotation on top, scRNA-seq and snRNA-seq side by side, `RdPu`.

Each factor is z-scored **across the populations within its arm**. That removes the baseline each
assay puts on a factor, so the two panels answer the same question — which populations use this
programme most — and are comparable to each other. Raw mean usage share is one argument away
(`mode='abs'`) but is dominated by whichever programmes carry the baseline mass: S8 reaches 0.56 in
the perilobular nuclei while mature adipose peaks near 0.1, which flattens every other row.

In [26]:
M = G.load_model(repo_path, tag='k16_lam5')
U, obs_f = M['U'], M['light'].obs.copy()
obs_f['label_cm'] = anno_c2m.reindex(obs_f.index)['taa_l3_cm'].values
print('arm:', dict(obs_f['arm'].value_counts()), '| unlabelled:',
      int(obs_f['label_cm'].isna().sum()))

PUB_FACTORS = {
    'S1': 'S1 hypoxia / mechano', 'S2': 'S2 lymphoid', 'S3': 'S3 interlobular',
    'S4': 'S4 proliferation', 'S5': 'S5 perilobular', 'S6': 'S6 IEG',
    'S7': 'S7 inflammatory', 'S8': 'S8 perilobular (nuclear)', 'S9': 'S9 interlobular',
    'S10': 'S10 ECM synthesis', 'S11': 'S11 hormone / vascular', 'S12': 'S12 inflammatory stress',
    'S13': 'S13 myofibroblast', 'S14': 'S14 medullary', 'S15': 'S15 mature adipose',
    'S16': 'S16 modality',
}

fig = P.usage_heatmap(U, obs_f, label_col='label_cm', arms=('cells', 'nuclei'),
                      factor_labels=PUB_FACTORS, mode='z', cmap='RdPu',
                      width='one-half', cell_mm=5.0, min_cells=20,
                      suptitle='iNMF factor usage — z across populations')
P.save(fig, f'{FIGS}/Fig5_factorUsage_taa_l3_cm_{today}.pdf')
plt.show()

for arm in ('cells', 'nuclei'):
    t = P.usage_by_population(U, obs_f, 'label_cm', arm=arm, min_cells=20)
    t.T.to_csv(f'{RES}/Fig5_factorUsage_{arm}_{today}.csv')
    print(f'--- {arm}: mean usage share, populations x factors')
    print(t.round(3).to_string(), '\n')

arm: {'cells': np.int64(70258), 'nuclei': np.int64(14858)} | unlabelled: 0


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig5_factorUsage_taa_l3_cm_2026-10-28.{pdf}   130 x 107 mm
--- cells: mean usage share, populations x factors
               S1     S2     S3     S4     S5     S6     S7     S8     S9    S10    S11    S12    S13    S14    S15    S16
Fb-TGFB+    0.287  0.010  0.033  0.096  0.004  0.022  0.098  0.006  0.032  0.204  0.028  0.075  0.024  0.019  0.008  0.054
Fb-adipo    0.012  0.008  0.032  0.007  0.075  0.109  0.145  0.022  0.028  0.073  0.238  0.027  0.011  0.027  0.136  0.050
Fb-interlo  0.032  0.012  0.204  0.008  0.047  0.069  0.052  0.023  0.157  0.083  0.066  0.127  0.023  0.027  0.009  0.061
Fb-interm   0.040  0.009  0.048  0.005  0.049  0.087  0.386  0.021  0.013  0.024  0.085  0.090  0.015  0.043  0.012  0.071
Fb-med      0.013  0.023  0.019  0.009  0.117  0.138  0.029  0.093  0.010  0.040  0.070  0.006  0.160  0.239  0.008  0.026
Fb-perilo   0.013  0.016  0.043  0.008  0.207  0.11

--- nuclei: mean usage share, populations x factors
               S1     S2     S3     S4     S5     S6     S7     S8     S9    S10    S11    S12    S13    S14    S15    S16
Fb-TGFB+    0.385  0.012  0.005  0.057  0.013  0.014  0.076  0.018  0.033  0.220  0.007  0.058  0.031  0.019  0.006  0.047
Fb-adipo    0.009  0.007  0.010  0.009  0.042  0.101  0.040  0.084  0.017  0.021  0.245  0.010  0.005  0.009  0.286  0.106
Fb-interlo  0.056  0.013  0.180  0.010  0.022  0.066  0.020  0.063  0.236  0.055  0.021  0.074  0.011  0.012  0.004  0.158
Fb-interm   0.089  0.010  0.037  0.010  0.028  0.051  0.142  0.052  0.019  0.022  0.095  0.041  0.009  0.017  0.035  0.342
Fb-med      0.020  0.099  0.017  0.020  0.032  0.153  0.019  0.255  0.014  0.025  0.051  0.005  0.061  0.065  0.015  0.148
Fb-perilo   0.008  0.014  0.042  0.011  0.060  0.100  0.022  0.319  0.027  0.012  0.220  0.005  0.015  0.017  0.024  0.107 



## 8. Factors across the Xenium niches

From the cached donor × niche pseudobulk scores of `Xenium_Fb_factors_niche_v1.ipynb` (all cells)
and `Xenium_Fb_factors_fbOnly_v1.ipynb` (fibroblasts only). Nothing is recomputed here.

**Dot colour is the mean score; dot area is the fraction of donors that agree with it** (same side
of zero). With five sections a heatmap of the mean hides the only thing that decides whether a
niche effect is real, which is whether the sections say the same thing. A large dot means they all
do; a small one means the mean is an average over disagreement.

Scores are centred **within donor** across niches before averaging — the sections differ several-
fold in pseudobulk depth, and the question is which niche is high *within* a section.

**S1 has no fibroblast-only score.** In the all-cell notebook S1/TGFB failed the
fibroblast-readability gate (its best non-fibroblast type, EC-lymphatic, outscores its best
fibroblast state), so it was never carried into the fibroblast-restricted run. It appears in the
all-cell panels only, and a niche pattern for S1 there is not evidence about fibroblasts.

**S11 (hormone-responsive / vascular-tone) joins the selected set in v2.** Unlike S1 it passed the
fibroblast-readability gate, so it is carried into the fibroblast-only run and appears in all three
adult panels (`Fig6b`, `6c`, `6d`), with all three adult donors behind every niche.

**v3 revision (VK, 2026-10-28): the selected factors are regrouped, S10 joins them, and the panels
are redrawn in red-blue.** Rows run in three groups, in this order: inflammatory S7 / S12, fibrotic
S1 / S10, then adipogenic S11 / S15 (`SELECTED_GROUPS`). They are separated by a rule in the dotplots
and by a gap in the heatmap. The same grouping orders the factor columns of the key-TF panel in §10
(`Fig8a`). The regrouped panels carry `selectedGrouped` and `_RdBu` in their names, so they sit
beside the earlier `Fig6b`–`6d` files rather than replacing them. S1 is still absent from the
fibroblast-only panel (`Fig6c`), so its fibrotic group holds S10 alone.

In [27]:
SPATIAL = f'{REPO}/Spatial_analyses/results/xenium/fb_factor_spatial'
sc_all = pd.read_csv(f'{SPATIAL}/thyAgeing_xenium_fbFactors_niche_table1_scores.csv')
sc_fb  = pd.read_csv(f'{SPATIAL}/thyAgeing_xenium_fbFactors_fbOnly_table1_scores.csv')

ALL_FACTORS = [f'S{i}' for i in range(1, 17)]
# VK (2026-10-28, v3 rev): grouped and drawn in this order -- inflammatory, fibrotic, adipogenic.
# Also the column groups of the §10 key-TF panel (Fig8a), so the two figures read the same way.
SELECTED_GROUPS = {'Inflammatory': ['S7', 'S12'],
                   'Fibrotic':     ['S1', 'S10'],
                   'Adipogenic':   ['S11', 'S15']}
SELECTED = [f for fs in SELECTED_GROUPS.values() for f in fs]
CMAP_RB = plt.get_cmap('RdBu_r')           # VK: red-blue for the regrouped §8 and Fig8a panels


def present_groups(groups, have):
    """`groups` restricted to the members in `have`, empty groups dropped."""
    have = set(have)
    out = {g: [m for m in ms if m in have] for g, ms in groups.items()}
    return {g: ms for g, ms in out.items() if ms}


def mark_group_breaks(ax, groups):
    """Thin rule between consecutive row groups of a dotplot (rows at 0, 1, 2, ...)."""
    for b in np.cumsum([len(ms) for ms in groups.values()])[:-1]:
        ax.axhline(b - 0.5, color=P.thyAgeing_greys['grey3'], linewidth=0.4, zorder=0)
# VK's order (2026-08-30): cortex -> medulla -> the adipose/fibrotic axis -> vessel. Used by
# EVERY panel in this section, so the niche axis reads the same way across the figure set.
NICHE_ORDER = ['cortex_septal', 'cortex', 'medulla', 'medulla_hc',
               'adipose', 'fibrosis_adipose', 'fibrosis', 'vessel']

# patch pseudobulks -- H&E morphology patches grouped into niches, 5 sections
patches = sc_all[(sc_all['representation'] == 'patches')
                 & (sc_all['key'] == 'morpho_niche_l2')].copy()
patches['niche'] = patches['group'].astype(str)
patches_ad = patches[patches['age_group'] == 'adult'].copy()   # A67 / A70 / A72

# cell pseudobulks, adult sections only
cells_ad = sc_all[(sc_all['representation'] == 'cells') & (sc_all['key'] == 'morpho_niche_l2')
                  & (sc_all['age_group'] == 'adult')].copy()
cells_ad['niche'] = cells_ad['group'].astype(str)

# the same, restricted to fibroblasts
fb_ad = sc_fb[(sc_fb['niche_level'] == 'morpho_niche_l2') & (sc_fb['state'] == 'all_fb')
              & (sc_fb['age_group'] == 'adult')].copy()

print('patches : donors', sorted(patches['donor'].unique()),
      '| age groups', sorted(patches['age_group'].unique()))
print('patches, adult only:', sorted(patches_ad['donor'].unique()))
print('adult   : donors', sorted(cells_ad['donor'].unique()))
print('fb-only adult sources:', [f for f in SELECTED if f in set(fb_ad['source'])],
      ' <- S1 absent by design')

patches : donors ['A67', 'A70', 'A72', 'Z16', 'Z17'] | age groups ['adult', 'lpaed']
patches, adult only: ['A67', 'A70', 'A72']
adult   : donors ['A67', 'A70', 'A72']
fb-only adult sources: ['S7', 'S12', 'S10', 'S11', 'S15']  <- S1 absent by design


In [28]:
# --- Fig 6a: ALL 16 factors across niches, patch pseudobulks, all 5 sections
fig = P.niche_factor_dotplot(patches, ALL_FACTORS, niche_order=NICHE_ORDER,
                             labels=PUB_FACTORS, width='one-half', cell_mm=6.0,
                             suptitle='All iNMF factors across Xenium niches (H&E patches, 5 sections)')
P.save(fig, f'{FIGS}/Fig6a_nicheFactors_allFactors_patches_{today}.pdf')
plt.show()

Mx, _ = P.niche_factor_matrix(patches, ALL_FACTORS, niche_col='niche')
Mx.to_csv(f'{RES}/Fig6a_nicheFactors_allFactors_patches_{today}.csv')
Mx[[n for n in NICHE_ORDER if n in Mx.columns]].round(2)

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6a_nicheFactors_allFactors_patches_2026-10-28.{pdf}   130 x 126 mm


niche,cortex_septal,cortex,medulla,medulla_hc,adipose,fibrosis_adipose,fibrosis,vessel
source,,,,,,,,
S1,-0.30,-0.31,-0.40,-0.01,0.10,0.36,0.24,0.31
S2,0.13,1.59,1.12,-0.59,-0.22,-0.93,-0.11,-0.99
S3,-0.57,-0.69,-1.01,-1.17,-0.10,1.83,1.09,0.63
S4,1.71,1.29,0.21,-2.34,-0.11,-0.29,0.31,-0.78
S5,-0.18,-0.84,-0.58,-1.08,0.40,1.01,0.49,0.78
S6,-0.42,-0.80,-0.45,0.24,0.10,0.56,0.22,0.55
S7,-0.60,-0.75,-0.91,-0.90,0.40,1.15,0.92,0.71
S8,-0.10,-0.27,-0.11,-0.14,0.20,0.01,-0.01,0.42
S9,-0.46,-0.62,-0.74,-0.85,-0.18,1.85,0.73,0.27


In [29]:
# --- Fig 6b/6c: the selected factors across the three ADULT sections, grouped, red-blue
for src, df, tag, name in [('cells', cells_ad, '6b', 'all cells'),
                           ('fb', fb_ad, '6c', 'fibroblasts only')]:
    groups = present_groups(SELECTED_GROUPS, df['source'])
    present = [f for fs in groups.values() for f in fs]
    fig = P.niche_factor_dotplot(df, present, niche_order=NICHE_ORDER, labels=PUB_FACTORS,
                                 width='one-half', cell_mm=7.0, cmap=CMAP_RB, keep_order=True,
                                 suptitle=f'Selected factors across adult Xenium niches — {name}')
    mark_group_breaks(fig.axes[0], groups)
    P.save(fig, f'{FIGS}/Fig{tag}_nicheFactors_selectedGrouped_adult_{src}_RdBu_{today}.pdf')
    plt.show()
    Mx, _ = P.niche_factor_matrix(df, present, niche_col='niche')
    Mx.to_csv(f'{RES}/Fig{tag}_nicheFactors_selectedGrouped_adult_{src}_{today}.csv')
    print(name, '| groups:', groups)
    print(Mx[[n for n in NICHE_ORDER if n in Mx.columns]].round(2).to_string(), '\n')

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6b_nicheFactors_selectedGrouped_adult_cells_RdBu_2026-10-28.{pdf}   130 x 72 mm


all cells | groups: {'Inflammatory': ['S7', 'S12'], 'Fibrotic': ['S1', 'S10'], 'Adipogenic': ['S11', 'S15']}
niche   cortex_septal  cortex  medulla  medulla_hc  adipose  fibrosis_adipose  fibrosis  vessel
source                                                                                         
S7              -0.76   -0.96    -1.45       -1.77     0.70              1.71      1.47    1.06
S12             -0.63   -0.46    -0.94       -0.73     0.10              1.20      0.98    0.48
S1              -0.41   -0.44    -0.57       -0.26     0.26              0.44      0.48    0.50
S10             -0.64   -0.75    -0.94       -0.63     0.49              1.16      0.63    0.67
S11             -0.40   -0.94    -1.50       -3.19     2.10              1.80      1.17    0.97
S15             -1.16   -0.11    -1.48       -2.09     2.86              1.67      0.52   -0.22 

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6c_nicheFactors_s

### 8.1 The selected factors across adult patches — plain heatmap

`Fig6d`: S7 / S12 | S1 / S10 | S11 / S15 across the eight niches, from the **patch** pseudobulks of
the three adult sections (A67, A70, A72). Every factor × niche cell is backed by all three donors,
so no cell needs a donor-count annotation. The cell asserts this, because the grouped heatmap
draws no counts.

This is the same quantity the dotplots show, drawn plainly: colour is the mean score, centred
within donor across niches before averaging, because the sections differ several-fold in patch
depth and the question is which niche is high *within* a section.

Two versions, as with the dotplots:

- **raw** (no suffix) — one shared colour scale, so the factors are comparable to each other.
- **`_rowScaled`** — each row divided by its own |max|. The factors differ several-fold in range,
  so on the shared scale the smallest is nearly flat; this version shows each factor's niche
  *pattern* and deliberately discards the magnitude.

Note what the plain heatmap drops relative to `Fig6b`: the dot area there was the fraction of
donors agreeing with the mean. Three sections can average to a confident-looking colour out of
2-vs-1 disagreement, so read `Fig6b` alongside this before calling any single square a result.

In [30]:
M_ad, per_donor_ad = P.niche_factor_matrix(patches_ad, SELECTED, niche_col='niche')
M_ad = M_ad[[n for n in NICHE_ORDER if n in M_ad.columns]]
groups_ad = present_groups(SELECTED_GROUPS, M_ad.index)
# the grouped heatmap draws no donor counts, so the claim it relies on is checked here instead
n_don = per_donor_ad.notna().groupby(level=0).sum().reindex(index=M_ad.index, columns=M_ad.columns)
print('donors behind each factor x niche cell:', int(n_don.values.min()), '-', int(n_don.values.max()))
assert n_don.values.min() == n_don.values.max() == 3, 'a cell is not backed by all three donors'

CB_6D = 'factor score\n(centred within donor)'
for row_norm, sfx in [(False, ''), (True, '_rowScaled')]:
    # row_norm as niche_factor_heatmap's: each factor divided by its own |max|
    Mp = M_ad.div(M_ad.abs().max(axis=1).replace(0, np.nan), axis=0) if row_norm else M_ad
    fig = P.grouped_heatmap(Mp, row_groups=groups_ad, labels=PUB_FACTORS, cmap=CMAP_RB,
                            vlim=1.0 if row_norm else None, group_labels=False,
                            cbar_label=(CB_6D.split('\n')[0] + '\n(scaled to each factor\u2019s |max|)'
                                        if row_norm else CB_6D),
                            width='single', cell_mm=7.0,
                            suptitle='Selected factors across adult niches (H&E patches)')
    P.save(fig, f'{FIGS}/Fig6d_nicheFactors_selectedGrouped_adult_patches{sfx}_RdBu_{today}.pdf')
    plt.show()

M_ad.to_csv(f'{RES}/Fig6d_nicheFactors_selectedGrouped_adult_patches_{today}.csv')
M_ad.round(2)

donors behind each factor x niche cell: 3 - 3


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6d_nicheFactors_selectedGrouped_adult_patches_RdBu_2026-10-28.{pdf}   88 x 61 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6d_nicheFactors_selectedGrouped_adult_patches_rowScaled_RdBu_2026-10-28.{pdf}   88 x 61 mm


niche,cortex_septal,cortex,medulla,medulla_hc,adipose,fibrosis_adipose,fibrosis,vessel
source,,,,,,,,
S7,-0.80,-0.93,-1.45,-1.77,0.70,1.74,1.47,1.03
S12,-0.66,-0.45,-0.93,-0.72,0.07,1.26,0.96,0.46
S1,-0.42,-0.44,-0.58,-0.28,0.24,0.54,0.45,0.48
S10,-0.75,-0.77,-1.04,-0.77,0.50,1.40,0.75,0.68
S11,-0.50,-0.92,-1.55,-3.26,2.15,1.96,1.22,0.91
S15,-1.48,-0.09,-1.74,-2.46,3.20,2.23,0.76,-0.42


In [31]:
# VK 2026-10-28: the same panel in the grouped order, but in the ORIGINAL colour scale
# (niche_factor_heatmap's default blue-magenta) and without the gaps between the factor groups
for row_norm, sfx in [(False, ''), (True, '_rowScaled')]:
    fig = P.niche_factor_heatmap(M_ad, per_donor=per_donor_ad, niche_order=NICHE_ORDER,
                                 labels=PUB_FACTORS, width='single', cell_mm=7.0,
                                 row_norm=row_norm,
                                 suptitle='Selected factors across adult niches (H&E patches)')
    P.save(fig, f'{FIGS}/Fig6d_nicheFactors_selectedGrouped_adult_patches_noGap{sfx}_{today}.pdf')
    plt.show()

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6d_nicheFactors_selectedGrouped_adult_patches_noGap_2026-10-28.{pdf}   88 x 66 mm
-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6d_nicheFactors_selectedGrouped_adult_patches_noGap_rowScaled_2026-10-28.{pdf}   88 x 66 mm


### 8.2 Every signature, split by age group

All 23 sources — the 16 individual factors **and** the 7 pooled groups — across the eight niches,
one panel per age group.

**The cells arm is the primary version because it is the only one that has all three age groups.**
The patch arm has no early-paediatric sections at all (U48 and U51 have no transcript-level
patches), which is why VK's note says to use cells there. Rather than mixing representations
inside one comparison — a patch pseudobulk and a cell pseudobulk are different units, so an
epaed cells column next to an adult patches column would confound age with representation — the
two arms are drawn as two separate figure sets:

- `Fig6e…_cells` — **epaed / lpaed / adult**, 2 / 2 / 3 donors, all 8 niches present in each.
- `Fig6f…_patches` — **lpaed / adult only**, stated as such.

**Both scalings are cross-comparable between the age panels, which took some care:**

- **raw** — one shared `vlim` across all age groups, so a square means the same thing in every
  panel.
- **`_rowScaled`** — each source divided by its |max| taken over **all age groups pooled**, not
  within its own panel. Per-panel scaling would have made each age group its own reference and
  the three panels no longer comparable, which is the opposite of what this figure is for. The
  scaled version is needed because the per-source range spans ~35-fold (0.13 to 4.84), so on the
  shared scale most rows are washed out.

The 7 pooled groups are included for completeness but read them with the caveat the spatial work
already established: pooling hurts at Xenium resolution — `Inflammatory` (S7+S12) is not
fibroblast-readable while S7 and S12 each are, and `Profibrotic` (S10+S13) reads only because S10
carries it. Prefer the individual factors.

In [32]:
ALL_SOURCES = ALL_FACTORS + ['Perilobular', 'Interlobular', 'Medullary',
                             'Adipose', 'Inflammatory', 'Profibrotic', 'TGFB']
AGE_SPATIAL = ['epaed', 'lpaed', 'adult']
AGE_TITLE   = {'epaed': 'early paediatric', 'lpaed': 'late paediatric', 'adult': 'adult'}

cells_all = sc_all[(sc_all['representation'] == 'cells')
                   & (sc_all['key'] == 'morpho_niche_l2')].copy()
cells_all['niche'] = cells_all['group'].astype(str)

for nm, df in [('cells', cells_all), ('patches', patches)]:
    have = [a for a in AGE_SPATIAL if a in set(df['age_group'])]
    n = df.groupby('age_group', observed=True)['donor'].nunique().reindex(have)
    print(f'{nm:8s} age groups {have}  donors {n.tolist()}')

cells    age groups ['epaed', 'lpaed', 'adult']  donors [2, 2, 3]
patches  age groups ['lpaed', 'adult']  donors [2, 3]


In [33]:
def age_split_heatmaps(df, tag, rep_name):
    """One heatmap per age group, on scales that are comparable BETWEEN the panels."""
    mats, pdon = {}, {}
    for ag in AGE_SPATIAL:
        d = df[df['age_group'] == ag]
        if d.empty:
            continue
        mats[ag], pdon[ag] = P.niche_factor_matrix(d, ALL_SOURCES, niche_col='niche')

    vlim = max(float(np.nanmax(np.abs(M.values))) for M in mats.values())
    # ONE divisor per source, taken over every age group -- not per panel, or the panels stop
    # being comparable to each other
    denom = pd.concat(mats.values()).abs().max(axis=1).groupby(level=0).max()

    out = []
    for ag, M in mats.items():
        for scaled, sfx in [(False, ''), (True, '_rowScaled')]:
            Mp = M.div(denom.reindex(M.index).replace(0, np.nan), axis=0) if scaled else M
            cb = ('factor score\n(scaled to each source’s |max|,\nall ages pooled)' if scaled
                  else 'factor score\n(centred within donor)')
            fig = P.niche_factor_heatmap(
                Mp, per_donor=pdon[ag], niche_order=NICHE_ORDER, labels=PUB_FACTORS,
                width='single', cell_mm=6.0, vlim=1.0 if scaled else vlim, cbar_label=cb,
                suptitle=f'All signatures — {AGE_TITLE[ag]} ({rep_name})')
            P.save(fig, f'{FIGS}/Fig{tag}_nicheSignatures_allSources_{ag}_{rep_name}{sfx}_{today}.pdf')
            plt.show()
        out.append(M.assign(age_group=ag, representation=rep_name))
    tab = pd.concat(out)
    tab.to_csv(f'{RES}/Fig{tag}_nicheSignatures_allSources_{rep_name}_{today}.csv')
    print(f'{rep_name}: shared vlim {vlim:.2f} | per-source |max| '
          f'{denom.min():.2f}-{denom.max():.2f}')
    return tab


tab_cells   = age_split_heatmaps(cells_all, '6e', 'cells')
tab_patches = age_split_heatmaps(patches,   '6f', 'patches')

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6e_nicheSignatures_allSources_epaed_cells_2026-10-28.{pdf}   88 x 161 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6e_nicheSignatures_allSources_epaed_cells_rowScaled_2026-10-28.{pdf}   88 x 161 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6e_nicheSignatures_allSources_lpaed_cells_2026-10-28.{pdf}   88 x 161 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6e_nicheSignatures_allSources_lpaed_cells_rowScaled_2026-10-28.{pdf}   88 x 161 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6e_nicheSignatures_allSources_adult_cells_2026-10-28.{pdf}   88 x 161 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6e_nicheSignatures_allSources_adult_cells_rowScaled_2026-10-28.{pdf}   88 x 161 mm
cells: shared vlim 4.84 | per-source |max| 0.57-4.84


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6f_nicheSignatures_allSources_lpaed_patches_2026-10-28.{pdf}   88 x 161 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6f_nicheSignatures_allSources_lpaed_patches_rowScaled_2026-10-28.{pdf}   88 x 161 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6f_nicheSignatures_allSources_adult_patches_2026-10-28.{pdf}   88 x 161 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig6f_nicheSignatures_allSources_adult_patches_rowScaled_2026-10-28.{pdf}   88 x 161 mm
patches: shared vlim 4.75 | per-source |max| 0.55-4.75


## 9. Fibroblast → vascular crosstalk (Xenium)

Five ligand–receptor pairs, drawn from the **Xenium** object with the **updated** fibroblast
annotation: `VEGFA→KDR`, `FGF2→FGFR1`, `CXCL12→CXCR4`, `JAG1→NOTCH1`,
`SLIT2→ROBO4`. All ten genes are on the 5,043-gene panel, so nothing is dropped.

**Which annotation supplies which side.** Sources are the fibroblast/adipose states from
`taa_l3_cmx` — the new annotation, joined onto the Xenium cells by barcode (all 64,108 match).
Targets are the vascular states from `taa_l3_comb`, which the re-annotation did not touch. The two
are pasted into one `population` column, so a cell is either a source or a target, never both.

The updated annotation matters here: it more than doubles Fb-interlo (4,359 → 9,835 cells) and
adds `Fb-TGFB+`, which had no `taa_l3_comb` counterpart at all.

**This is a co-expression map, not a proven interaction.** It shows that a ligand is expressed by
one population and its receptor by another in the same tissue; it does not show the two cells are
in contact, and nothing here is a spatial-proximity test. Dot area is the detected fraction, colour
the mean over **all** cells in the group, **unscaled** — raw log-normalised, one shared scale across both
panels, because a ligand panel scaled to itself beside a receptor panel scaled to itself cannot be
read as a single figure.

Xenium detection is sparse per cell, so absolute fractions here run lower than the dissociated
equivalent; compare columns within this figure, not against the scRNA-seq version.

Written twice: the default 89 mm version, and a **56 x 42 mm** one with every text element at
6 pt for a fixed slot in a composite.

In [34]:
import xenium_steroid_scoring as S

XPQ = f'{REPO}/Spatial_analyses/data/xenium/thyAgeing_xenium_crosstalk_lognorm.parquet'
LIGANDS   = ['VEGFA', 'FGF2', 'CXCL12', 'JAG1', 'SLIT2']
RECEPTORS = ['KDR', 'FGFR1', 'CXCR4', 'NOTCH1', 'ROBO4']
SRC_ORDER = ['Fb-interlo', 'Fb-perilo', 'Fb-med', 'Fb-interm', 'Fb-adipo', 'Fb-TGFB+', 'Adipocyte']
TGT_ORDER = ['EC-art', 'EC-cap', 'EC-cap_lipid', 'EC-ven', 'EC-peri', 'EC-lymphatic',
             'SMC', 'Pericyte']

E = pd.read_parquet(XPQ)
xobs = S.load_annotations(verbose=False)
cmx_x = pd.read_csv(CMX_ANNO, index_col=0, usecols=['Unnamed: 0', 'taa_l3_cmx'])
cmx_x = cmx_x[cmx_x.index.astype(str).str.startswith('xenium_')]

# sources from the NEW annotation, targets from the untouched vascular calls
pop = xobs['taa_l3_comb'].astype(str).where(xobs['taa_l3_comb'].astype(str).isin(TGT_ORDER))
pop.loc[cmx_x.index] = cmx_x['taa_l3_cmx'].astype(str)
xobs['population'] = pop
E = E.reindex(xobs.index)
print('cells with a source/target label:', int(xobs['population'].notna().sum()))
print(xobs['population'].value_counts().reindex(SRC_ORDER + TGT_ORDER).to_string())

cells with a source/target label: 179656
population
Fb-interlo       9835
Fb-perilo       30115
Fb-med           8771
Fb-interm        7340
Fb-adipo         1668
Fb-TGFB+          184
Adipocyte        6195
EC-art           7456
EC-cap          46593
EC-cap_lipid     1022
EC-ven          26374
EC-peri           766
EC-lymphatic     1997
SMC             16677
Pericyte        14663


In [35]:
# mean over ALL cells in the group, not only expressing ones. On Xenium these genes are detected
# in 0.5-7% of cells, so a mean restricted to expressing cells is essentially the value of a
# single transcript and comes out flat at ~3-4 for every population -- the colour then carries no
# information at all. Averaging in the zeros is what makes the colour vary, and it is also what
# scanpy's dotplot does by default.
mean_l, frac_l = P.expr_summary(E, xobs['population'], LIGANDS, SRC_ORDER,
                                mean_only_expressed=False)
mean_r, frac_r = P.expr_summary(E, xobs['population'], RECEPTORS, TGT_ORDER,
                                mean_only_expressed=False)

fig = P.crosstalk_dotplot(
    [dict(title='Source', mean=mean_l, frac=frac_l, labels='left'),
     dict(title='Target', mean=mean_r, frac=frac_r, labels='right')],
    width='single', cell_mm=5.0,
    suptitle='Fibroblast and Adipose/Vasculature crosstalk')
P.save(fig, f'{FIGS}/Fig7_crosstalk_fbVascular_xenium_{today}.pdf')
plt.show()

# compact version for a fixed slot: exactly 56 x 42 mm, all text 6 pt. At that size the panel
# titles and the figure title cannot both fit above five rows of dots, so the suptitle is dropped
# and belongs in the composite instead.
fig = P.crosstalk_dotplot(
    [dict(title='Source', mean=mean_l, frac=frac_l, labels='left'),
     dict(title='Target', mean=mean_r, frac=frac_r, labels='right')],
    width=56.0, height_mm=42.0, fontsize=6.0, legend_mm=9.0,
    cell_mm=5.0, max_dot_mm=3.4, suptitle=None)
P.save(fig, f'{FIGS}/Fig7_crosstalk_fbVascular_xenium_56x42_{today}.pdf')
plt.show()

pd.concat({'ligand_mean': mean_l, 'ligand_frac': frac_l,
           'receptor_mean': mean_r, 'receptor_frac': frac_r}
          ).to_csv(f'{RES}/Fig7_crosstalk_fbVascular_xenium_{today}.csv')
mean_l.round(2)

findfont: Font family ['cursive'] not found. Falling back to DejaVu Sans.


findfont: Generic family 'cursive' not found because none of the following families were found: Apple Chancery, Textile, Zapf Chancery, Sand, Script MT, Felipa, cursive


   crosstalk: 88 x 54 mm | pitch 4.52 x 4.52 mm | dot<=3.40 mm | text 5 pt
-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig7_crosstalk_fbVascular_xenium_2026-10-28.{pdf}   88 x 54 mm


   crosstalk: 56 x 42 mm | pitch 2.18 x 2.90 mm | dot<=2.01 mm | text 6 pt
-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig7_crosstalk_fbVascular_xenium_56x42_2026-10-28.{pdf}   56 x 42 mm


,Fb-interlo,Fb-perilo,Fb-med,Fb-interm,Fb-adipo,Fb-TGFB+,Adipocyte
VEGFA,0.25,0.26,0.43,0.31,0.51,0.29,1.17
FGF2,0.17,0.14,0.14,0.16,0.23,0.08,0.25
CXCL12,1.71,1.92,1.53,1.75,1.65,1.17,0.25
JAG1,0.28,0.33,0.39,0.21,0.23,0.30,0.28
SLIT2,0.43,0.41,0.45,0.21,0.23,0.27,0.13


## 10. SCENIC+ eRegulon enrichment across the iNMF factors — rank test, AUC

v3 replaces v2's ULM score with a **one-sided rank test with the AUC as effect size**, keeps v2's
per-gene statistic, and calibrates the test against an **expression-matched permutation null**:

- **statistic** — the per-gene z across the 16 factors of the shared metagenes `W` (each factor
  normalised to sum 1, then each gene z-scored across factors; `G.gene_zscores`). Not raw `W`: raw
  loadings track gene abundance by different amounts in different factors (r = 0.05 in S15 up to
  0.56 in S7, from nb09), and the key TFs' targets are 1.2–5.5× more abundant than the basis
  average, so a test on raw loadings partly ranks eRegulons by their targets' abundance. The
  raw-loadings version is still computed, as a diagnostic (§10.2).
- **test** — for one factor and one eRegulon, the basis genes split into targets and everything
  else, with no cutoff on the value. `AUC = U / (n₁ n₂)` from the Mann–Whitney U of targets vs the
  rest: the probability that a random target outranks a random non-target (0.5 = no enrichment).
  Unlike the ULM t, the AUC does not grow with regulon size, so a 22-target and a 323-target
  eRegulon sit on one scale.
- **null** — the per-gene statistic shuffled 1,000 times **within 20 abundance bins**, so each gene
  receives the value of a gene of similar mean expression, and the AUC recomputed each time.
  Reported: `dAUC` = AUC − null mean AUC (the effect, drawn on the heatmaps); `z` = dAUC / null SD;
  one-sided `p` from `z` (enrichment only, as `alternative='greater'`); the empirical `p_emp`.
- **multiple testing** — BH across **all** factor × eRegulon tests in the run, not per factor.
- `p_mwu` — the plain one-sided Mann–Whitney p on the same statistic. It assumes the genes are
  exchangeable, so it is kept only to show how much the matched null changes the calibration.

**The matched null is implemented here, not taken from `steroid_de_scoring.null_for`.** That
function shuffles within bins in the *bin-sorted* gene order and then scores the result against an
adjacency in the *original* gene order without mapping it back, so most genes receive a value from
another bin and its null is effectively unmatched (this affects v2's §10 and nb09). The scoring
cell measures that fraction and asserts that the permutation used here hands every gene a value
from its own bin.

**Network: SCENIC+ v2, activating `_+_+` eRegulons only**, built exactly as in v2.

- **Fig 8a** — key TFs grouped by programme × the factors grouped as in §8: S7 / S12, S1 / S10,
  S11 / S15 (dAUC, red-blue; the v3 revision, drawn after §10.1).
- **Fig 8b** — the original key-TF list across all 16 factors.
- **Fig 8c** — other inflammatory / adipogenic / profibrotic TFs enriched in the selected factors.
- **Fig 8d** — the key-TF panel under the raw-loadings recipe (AUC − 0.5, no null), for comparison.

8b–8d share one colour scale (both quantities are AUC differences); 8a has its own. `*` = FDR < 0.05,
BH across all tests. An eRegulon scoring in a factor means the factor's loadings are concentrated on genes SCENIC+
linked to that TF — enrichment of a target set, not a measurement of the TF's own expression or
binding in fibroblasts.

In [36]:
from scipy.stats import norm, spearmanr, rankdata, mannwhitneyu
from statsmodels.stats.multitest import multipletests
import steroid_de_scoring as SD     # bin_ids only -- the permutation itself is implemented below
                                    # (`S` is already xenium_steroid_scoring, from §9)
import logging
# decoupler (imported by SD) configures logging so that fontTools prints its font-subsetting INFO
# lines into the notebook on every PDF save
logging.getLogger('fontTools').setLevel(logging.WARNING)

SCENIC_V2 = f'{REPO}/Nuclei_multiome/eregulon_metadata_full_20runs_combined_min3recall_v2.csv'
INMF_DIR  = f'{repo_path}/data/nmf_joint/inmf'
COLLECTRI_RES = f'{repo_path}/results/factor_tf_activity/collectri_factor_activity_k16_lam5.csv'

N_BINS = 20       # abundance bins the permutation is matched within
TMIN   = 10       # targets an eRegulon needs inside the basis to be tested (as nb09 / v2)
N_PERM = 1000     # matched permutations per factor
FDR    = 0.05     # the * on every heatmap; BH across ALL tests

# VK's key inflammatory / fibrotic / adipogenic TFs (2026-10-28), in the order given
KEY_TFS = ['PPARG', 'BCL6', 'TWIST1', 'AR', 'EBF2', 'CEBPB', 'MYC', 'FOSL1', 'GATAD2A',
           'CREB3L1', 'NFIX', 'ATF4', 'ATF6']
TF_FACTORS = ['S1', 'S7', 'S10', 'S11', 'S15']

meta = pd.read_csv(SCENIC_V2, usecols=['TF', 'Gene', 'TF2G_sign', 'R2G_sign',
                                       'n_called_triplet_full'])
assert meta['n_called_triplet_full'].min() >= 3, 'expected the min3recall table'
print('v2 triplets by class:', (meta['TF2G_sign'] + '_' + meta['R2G_sign']).value_counts().to_dict())

# activating eRegulons only; one edge per TF x gene -- a gene reached through several regions is
# still one target
act = (meta[(meta['TF2G_sign'] == '+') & (meta['R2G_sign'] == '+')]
       .drop_duplicates(['TF', 'Gene']).reset_index(drop=True))
n_genes = act.groupby('TF')['Gene'].size()
tf_of = {f'{tf}_+_+_({n}g)': tf for tf, n in n_genes.items()}
src_of = {tf: s for s, tf in tf_of.items()}
net_sc = pd.DataFrame({'source': act['TF'].map(src_of), 'target': act['Gene'], 'weight': 1.0})
assert net_sc['source'].notna().all()
net_sc.to_csv(f'{RES}/Fig8_scenicplusV2_net_activating_{today}.csv', index=False)
print(f'activating eRegulons: {len(tf_of)} TFs, {len(net_sc):,} TF-gene edges, '
      f'{net_sc["target"].nunique():,} target genes')
del meta, act

v2 triplets by class: {'+_+': 352608, '-_+': 4154}


activating eRegulons: 244 TFs, 87,734 TF-gene edges, 12,152 target genes


In [37]:
W = pd.read_csv(f'{INMF_DIR}/W_k16_lam5.csv', index_col=0)          # 16 factors x 3,671 genes
gene_mean = pd.read_csv(f'{INMF_DIR}/gene_mean_expr.csv', index_col=0).iloc[:, 0]

_, Zg = G.gene_zscores(W)
Zg = Zg.loc[G.natural_factor_order(Zg.index), sorted(Zg.columns)]
universe = np.array(Zg.columns)
abundance = gene_mean.reindex(universe).to_numpy()
assert np.isfinite(abundance).all(), 'gene_mean does not cover the basis'
assert np.isfinite(Zg.to_numpy()).all(), 'NaN in the per-gene z'
print(f'statistic: {Zg.shape[0]} factors x {Zg.shape[1]:,} genes')

# coverage -- what truncation to the basis leaves of each eRegulon
in_basis = net_sc[net_sc['target'].isin(set(universe))]
cov = pd.DataFrame({'tf': pd.Series(tf_of),
                    'n_total': net_sc.groupby('source').size(),
                    'n_in_basis': in_basis.groupby('source').size()})
cov['n_in_basis'] = cov['n_in_basis'].fillna(0).astype(int)
cov['frac_retained'] = (cov['n_in_basis'] / cov['n_total']).round(3)
cov['scored'] = cov['n_in_basis'] >= TMIN
cov.rename_axis('source').to_csv(f'{RES}/Fig8_scenicplusV2_coverage_{today}.csv')
print(f'edges inside the basis: {len(in_basis):,} of {len(net_sc):,} '
      f'({len(in_basis) / len(net_sc):.1%})')
print(f'eRegulons tested (>= {TMIN} targets in the basis): {int(cov["scored"].sum())} of {len(cov)}')
missing = [tf for tf in KEY_TFS if tf not in set(cov['tf'])]
print('key TFs with no activating eRegulon in v2:', missing or 'none')
cov.rename_axis('source').reset_index().set_index('tf').reindex(KEY_TFS)

statistic: 16 factors x 3,671 genes
edges inside the basis: 18,370 of 87,734 (20.9%)
eRegulons tested (>= 10 targets in the basis): 160 of 244
key TFs with no activating eRegulon in v2: none


,source,n_total,n_in_basis,frac_retained,scored
tf,,,,,
PPARG,PPARG_+_+_(122g),122,52,0.426,True
BCL6,BCL6_+_+_(216g),216,77,0.356,True
TWIST1,TWIST1_+_+_(79g),79,44,0.557,True
AR,AR_+_+_(50g),50,22,0.440,True
EBF2,EBF2_+_+_(390g),390,114,0.292,True
CEBPB,CEBPB_+_+_(699g),699,128,0.183,True
MYC,MYC_+_+_(1188g),1188,241,0.203,True
FOSL1,FOSL1_+_+_(852g),852,323,0.379,True
GATAD2A,GATAD2A_+_+_(536g),536,131,0.244,True


In [38]:
# --- target membership of the tested eRegulons over the (sorted) basis
sources = sorted(cov.index[cov['scored']])
gi = pd.Series(np.arange(len(universe)), index=universe)
si = pd.Series(np.arange(len(sources)), index=sources)
sub = in_basis[in_basis['source'].isin(set(sources))]
Mem = np.zeros((len(universe), len(sources)))
Mem[gi[sub['target']].to_numpy(), si[sub['source']].to_numpy()] = 1.0
n1 = Mem.sum(axis=0)
n2 = len(universe) - n1
assert (n1 == cov.loc[sources, 'n_in_basis'].to_numpy()).all()


def auc_from_ranks(R):
    """(k x genes) ranks -> (k x eRegulons) AUC = U / (n1 n2), U from the targets' rank sum."""
    return (R @ Mem - n1 * (n1 + 1) / 2) / (n1 * n2)


# --- the matched permutation
bins = SD.bin_ids(abundance, n_bins=N_BINS)
order = np.argsort(bins, kind='stable')
g_sorted = bins[order].astype(np.float64)


def matched_perm(v, rng, k):
    """k permutations of v in which every gene receives the value of a gene from its OWN
    abundance bin."""
    keys = g_sorted[None, :] + rng.random((k, g_sorted.size))   # sorting keeps bin blocks intact
    shuffled = np.asarray(v, dtype=np.float64)[order][np.argsort(keys, axis=1, kind='stable')]
    out = np.empty_like(shuffled)
    out[:, order] = shuffled          # back to gene order -- the step SD.null_for leaves out
    return out


# self-test: permuting the bin labels themselves must hand every gene back its own bin
ok = np.array_equal(matched_perm(bins, np.random.default_rng(0), 5),
                    np.tile(bins.astype(np.float64), (5, 1)))
assert ok, 'the matched permutation moved a value across abundance bins'
# the same bin vector through SD.null_for's construction (shuffled in sorted order, never mapped back)
_keys = g_sorted[None, :] + np.random.default_rng(0).random((5, g_sorted.size))
_unmapped = bins[order][np.argsort(_keys, axis=1, kind='stable')]
print(f'genes receiving a value from ANOTHER abundance bin -- matched_perm: 0.0% | '
      f'SD.null_for construction: {(_unmapped != bins[None, :]).mean():.1%}')

genes receiving a value from ANOTHER abundance bin -- matched_perm: 0.0% | SD.null_for construction: 94.1%


In [39]:
Wb = W.loc[Zg.index, universe].to_numpy()          # raw loadings, for the §10.2 diagnostic
tgt = [Mem[:, j] > 0 for j in range(len(sources))]
rng = np.random.default_rng(0)
rows = []
for i, f in enumerate(Zg.index):
    y = Zg.loc[f].to_numpy()
    R = rankdata(y)
    auc = auc_from_ranks(R[None, :])[0]
    # permuting the values within bins and re-ranking == permuting the ranks within bins
    null = np.vstack([auc_from_ranks(matched_perm(R, rng, min(200, N_PERM - s)))
                      for s in range(0, N_PERM, 200)])
    mu, sd = null.mean(axis=0), null.std(axis=0, ddof=1)
    z = (auc - mu) / np.where(sd > 0, sd, np.nan)
    mw = [mannwhitneyu(y[t], y[~t], alternative='greater') for t in tgt]
    mw_raw = [mannwhitneyu(Wb[i][t], Wb[i][~t], alternative='greater') for t in tgt]
    rows.append(pd.DataFrame({
        'factor': f, 'source': sources, 'tf': [tf_of[s] for s in sources],
        'n_targets': n1.astype(int), 'auc': auc, 'null_auc_mean': mu, 'null_auc_sd': sd,
        'dAUC': auc - mu, 'z': z, 'p': norm.sf(z),
        'p_emp': (1 + (null >= auc[None, :]).sum(axis=0)) / (N_PERM + 1),
        'auc_mwu': np.array([m.statistic for m in mw]) / (n1 * n2),
        'p_mwu': [m.pvalue for m in mw],
        'auc_raw': np.array([m.statistic for m in mw_raw]) / (n1 * n2),
        'p_raw': [m.pvalue for m in mw_raw]}))
    print(f'[{i + 1}/{len(Zg.index)}] {f}', flush=True)

res_sc = pd.concat(rows, ignore_index=True)
# the rank-sum AUC used for the null must be scipy's Mann-Whitney AUC
np.testing.assert_allclose(res_sc['auc'], res_sc['auc_mwu'], rtol=0, atol=1e-9)
res_sc = res_sc.drop(columns='auc_mwu')
res_sc['name'] = res_sc['factor'].map(PUB_FACTORS)
res_sc['padj'] = multipletests(res_sc['p'], method='fdr_bh')[1]            # BH across ALL tests
res_sc['padj_mwu'] = multipletests(res_sc['p_mwu'], method='fdr_bh')[1]
res_sc['padj_raw'] = multipletests(res_sc['p_raw'], method='fdr_bh')[1]
res_sc['pct_dAUC'] = res_sc.groupby('factor')['dAUC'].rank(pct=True) * 100
res_sc.to_csv(f'{RES}/Fig8_scenicplusV2_MWU_factorEnrichment_allFactors_{today}.csv', index=False)

dev = res_sc['null_auc_mean'] - 0.5
print(f'{len(sources)} eRegulons x {len(Zg.index)} factors = {len(res_sc):,} tests, BH across all')
print(f'matched-null mean AUC - 0.5 (what abundance alone buys): median {dev.median():+.3f}, '
      f'5-95% {dev.quantile(.05):+.3f} to {dev.quantile(.95):+.3f}')
print(f'null AUC sd: median {res_sc["null_auc_sd"].median():.3f}')
for col, nm in [('padj', 'per-gene z, matched null'), ('padj_mwu', 'per-gene z, plain MWU'),
                ('padj_raw', 'raw loadings, plain MWU')]:
    print(f'  FDR < {FDR}  {nm:26s}: {int((res_sc[col] < FDR).sum()):5d} of {len(res_sc):,}')

[1/16] S1


[2/16] S2


[3/16] S3


[4/16] S4


[5/16] S5


[6/16] S6


... (7 outputs left out)

[14/16] S14


[15/16] S15


[16/16] S16


160 eRegulons x 16 factors = 2,560 tests, BH across all
matched-null mean AUC - 0.5 (what abundance alone buys): median -0.002, 5-95% -0.083 to +0.055
null AUC sd: median 0.031
  FDR < 0.05  per-gene z, matched null  :   441 of 2,560
  FDR < 0.05  per-gene z, plain MWU     :   456 of 2,560
  FDR < 0.05  raw loadings, plain MWU   :   946 of 2,560


In [40]:
# Context: what tops each selected factor, so the key TFs can be placed against it
top = (res_sc[res_sc['factor'].isin(TF_FACTORS)].sort_values('dAUC', ascending=False)
       .groupby('factor').head(12))
for f in TF_FACTORS:
    t = top[top['factor'] == f]
    print(f'--- {PUB_FACTORS[f]}\n    ' + ', '.join(
        f'{tf} {d:+.3f}{"*" if q < FDR else ""}' for tf, d, q in zip(t['tf'], t['dAUC'], t['padj'])))

--- S1 hypoxia / mechano
    HMGA2 +0.419*, GATAD2A +0.374*, CREB3L1 +0.283*, TEAD4 +0.263*, HIC1 +0.238*, FOSL1 +0.236*, NFIX +0.230*, BNC2 +0.193*, AHCTF1 +0.179, HIF1A +0.173*, ELK3 +0.170*, ZEB2 +0.157*
--- S7 inflammatory
    AHCTF1 +0.199, RORB +0.198*, FOSL1 +0.157*, GATAD2A +0.156*, RFX2 +0.152*, CEBPB +0.139*, HMGA2 +0.138*, TEAD4 +0.130*, HEYL +0.121, HIF1A +0.120*, NR2F2 +0.109*, MXD3 +0.100*
--- S10 ECM synthesis
    CREB3L1 +0.365*, MAZ +0.245*, NFIX +0.241*, ATF4 +0.239*, NFIC +0.220*, HIC1 +0.197*, MYOG +0.173*, NFE2L1 +0.171*, E2F1 +0.163*, E2F3 +0.156*, TFDP1 +0.142*, HES6 +0.125
--- S11 hormone / vascular
    BCL6 +0.265*, AR +0.262*, EBF3 +0.204*, PPARG +0.181*, MEIS1 +0.171*, EBF2 +0.160*, ZBTB16 +0.128, MYBL2 +0.127*, NFIA +0.127*, TWIST1 +0.123*, E2F3 +0.123*, IKZF2 +0.123*
--- S15 mature adipose
    PPARG +0.250*, BCL6 +0.246*, TWIST1 +0.196*, ATF6 +0.156*, CEBPA +0.148, SOX17 +0.137*, ZNF654 +0.132, ZBTB16 +0.129, EBF2 +0.129*, PBX4 +0.115, HEYL +0.112, PRDM16 +

### 10.1 Other inflammatory, adipogenic and profibrotic TFs in the selected factors

VK's question: which TFs **not** on the key list are enriched in S1 / S7 / S10 / S11 / S15 and
belong to one of the three programmes? Two steps, kept apart so the second can be argued with:

1. **data** — eRegulons in the top `100 − CAND_PCT` % of at least one selected factor by `dAUC`,
   with FDR < 0.05 (BH across all tests; the test is one-sided, so every hit is an enrichment);
2. **biology** — of those, the TFs in `TF_CLASSES`, a literature-based list curated in this cell
   and **not derived from the data**. It is deliberately conservative: lineage TFs and factors
   with mainly anti-fibrotic or anti-adipogenic roles (FLI1, TCF21, GATA2/3, FOXO1) are left out.
   Edit it freely. UPR / stress TFs (XBP1, DDIT3) sit with *inflammatory*, as ATF4/ATF6 do on the
   key list. A TF can carry more than one class.

Every hit that step 2 drops is printed too, so the curated list cannot hide anything — read that
table before concluding a programme has no other TF.

In [41]:
CAND_PCT = 90
TF_CLASSES = {
    'inflammatory': ['NFKB1', 'NFKB2', 'RELA', 'RELB', 'REL', 'BCL3', 'STAT1', 'STAT2', 'STAT3',
                     'STAT4', 'STAT6', 'IRF1', 'IRF3', 'IRF5', 'IRF7', 'IRF9', 'CEBPD', 'ATF3',
                     'JUN', 'JUNB', 'JUND', 'FOS', 'FOSB', 'FOSL2', 'EGR1', 'EGR3', 'NR4A1',
                     'NR4A2', 'NR4A3', 'MAFF', 'NFIL3', 'BATF', 'HIF1A', 'NFE2L2', 'KLF6',
                     'XBP1', 'DDIT3'],
    'adipogenic':   ['CEBPA', 'CEBPD', 'EBF1', 'ZNF423', 'KLF4', 'KLF5', 'KLF9', 'KLF15',
                     'SREBF1', 'NR1H3', 'RXRA', 'NR3C1', 'PRDM16', 'MLXIPL', 'CREB1', 'ZNF638',
                     'TCF7L1'],
    'profibrotic':  ['TWIST2', 'SNAI1', 'SNAI2', 'ZEB1', 'ZEB2', 'PRRX1', 'SMAD2', 'SMAD3',
                     'SMAD4', 'RUNX1', 'TEAD1', 'TEAD2', 'TEAD3', 'TEAD4', 'SRF', 'MRTFA',
                     'MRTFB', 'NFATC1', 'NFATC2', 'NFATC4', 'GLI1', 'GLI2', 'MEOX1', 'MEOX2',
                     'SOX9', 'EGR1', 'EGR2', 'ETS1', 'CREB3L2', 'HIF1A', 'FOSL2', 'STAT3',
                     'JUNB', 'KLF5'],
}
ABBR = {'inflammatory': 'I', 'adipogenic': 'A', 'profibrotic': 'F'}
tf_class = {}
for cls, tfs in TF_CLASSES.items():
    for tf in tfs:
        tf_class.setdefault(tf, []).append(ABBR[cls])
cls_of = lambda tf: ','.join(tf_class.get(tf, [])) or '-'
print('curated TFs with an activating v2 eRegulon tested here:',
      sum(tf in set(res_sc['tf']) for tf in tf_class), 'of', len(tf_class))

hit = res_sc[res_sc['factor'].isin(TF_FACTORS) & (res_sc['padj'] < FDR)
             & (res_sc['pct_dAUC'] >= CAND_PCT)]
hits_d = hit.pivot(index='tf', columns='factor', values='dAUC').reindex(columns=TF_FACTORS)
hits = pd.DataFrame({'class': [cls_of(t) for t in hits_d.index],
                     'on_key_list': hits_d.index.isin(KEY_TFS),
                     'n_factors': hits_d.notna().sum(axis=1).values,
                     'max_dAUC': hits_d.max(axis=1).round(3).values}, index=hits_d.index)
hits = hits.join(hits_d.round(3)).sort_values(['class', 'max_dAUC'], ascending=[True, False])
hits.to_csv(f'{RES}/Fig8c_scenicplusV2_MWU_topHits_selectedFactors_{today}.csv')

CAND = hits.index[(hits['class'] != '-') & ~hits['on_key_list']].tolist()
n_key = int(hits['on_key_list'].sum())
print(f'eRegulons in the top {100 - CAND_PCT}% of >= 1 selected factor (FDR < {FDR}): '
      f'{len(hits)}; on the key list {n_key}; NEW curated candidates {len(CAND)}')
print('\n=== NEW candidates (dAUC; blank = not in the top set for that factor) ===')
print(hits.loc[CAND].drop(columns='on_key_list').fillna('').to_string())
print('\n=== top hits NOT in TF_CLASSES (read these before trusting the list) ===')
print(hits[hits['class'] == '-'].drop(columns=['class', 'on_key_list']).fillna('')
      .sort_values('max_dAUC', ascending=False).to_string())

curated TFs with an activating v2 eRegulon tested here: 34 of 81
eRegulons in the top 10% of >= 1 selected factor (FDR < 0.05): 52; on the key list 12; NEW curated candidates 5

=== NEW candidates (dAUC; blank = not in the top set for that factor) ===
       class  n_factors  max_dAUC     S1     S7 S10 S11    S15
tf                                                            
PRDM16     A          2     0.105         0.099          0.105
TEAD4      F          2     0.263  0.263   0.13               
ZEB2       F          1     0.157  0.157                      
KLF6       I          1     0.117  0.117                      
HIF1A    I,F          2     0.173  0.173   0.12               

=== top hits NOT in TF_CLASSES (read these before trusting the list) ===
         n_factors  max_dAUC     S1     S7    S10    S11    S15
tf                                                             
HMGA2            2     0.419  0.419  0.138                     
GATAD2A          2     0.374  0.374  0.15

In [42]:
def tf_matrix(tfs, value, factors):
    """TFs (rows, in the given order) x factors; NaN where a TF has no tested eRegulon."""
    return (res_sc[res_sc['tf'].isin(tfs)].pivot(index='tf', columns='factor', values=value)
            .reindex(index=list(tfs), columns=list(factors)))


ALL_F = list(Zg.index)
D_key, Q_key, PCT_key, Z_key = (tf_matrix(KEY_TFS, v, ALL_F)
                                for v in ('dAUC', 'padj', 'pct_dAUC', 'z'))
D_cand, Q_cand = (tf_matrix(CAND, v, TF_FACTORS) for v in ('dAUC', 'padj'))
RAW_key = tf_matrix(KEY_TFS, 'auc_raw', TF_FACTORS) - 0.5          # the §10.2 diagnostic
QRAW_key = tf_matrix(KEY_TFS, 'padj_raw', TF_FACTORS)
not_scored = D_key.index[D_key.isna().all(axis=1)].tolist()
print('key TFs not tested (no eRegulon, or < TMIN targets in the basis):', not_scored or 'none')

pd.concat({'dAUC': D_key, 'z': Z_key, 'pct_dAUC': PCT_key, 'padj': Q_key}, axis=1).to_csv(
    f'{RES}/Fig8ab_scenicplusV2_MWU_keyTFs_allFactors_{today}.csv')
print('\nkey TFs, dAUC (AUC - matched-null mean AUC)')
print(D_key[TF_FACTORS].round(3).to_string())
print('\nkey TFs, FDR (BH across all tests)')
print(Q_key[TF_FACTORS].apply(lambda c: c.map(lambda q: f'{q:.1e}')).to_string())
print('\nkey TFs, percentile of dAUC among all tested eRegulons in the factor')
print(PCT_key[TF_FACTORS].round(0).to_string())

key TFs not tested (no eRegulon, or < TMIN targets in the basis): none

key TFs, dAUC (AUC - matched-null mean AUC)
factor      S1     S7    S10    S11    S15
tf                                        
PPARG   -0.028  0.016 -0.016  0.181  0.250
BCL6    -0.149 -0.052 -0.076  0.265  0.246
TWIST1  -0.033 -0.006  0.122  0.123  0.196
AR      -0.081 -0.003 -0.020  0.262 -0.040
EBF2    -0.137 -0.032 -0.013  0.160  0.129
CEBPB    0.020  0.139 -0.015  0.019  0.018
MYC      0.030  0.090 -0.050 -0.034 -0.004
FOSL1    0.236  0.157  0.012 -0.163 -0.040
GATAD2A  0.374  0.156  0.015 -0.105 -0.030
CREB3L1  0.283 -0.051  0.365 -0.033 -0.024
NFIX     0.230 -0.040  0.241 -0.057 -0.038
ATF4     0.134  0.058  0.239 -0.087  0.051
ATF6    -0.088 -0.037 -0.147 -0.090  0.156

key TFs, FDR (BH across all tests)
factor        S1       S7      S10      S11      S15
tf                                                  
PPARG    1.0e+00  8.0e-01  1.0e+00  1.0e-04  6.0e-09
BCL6     1.0e+00  1.0e+00  1.0e+00  3.1e-13 

In [43]:
# ONE scale for 8b-8d: all three show an AUC difference (8a is redrawn on its own scale below)
VLIM_TF = float(np.nanmax(np.abs(np.concatenate(
    [D_key.values.ravel(), D_cand.values.ravel(), RAW_key.values.ravel()]))))
CMAP_TF = P.CMAPS['blue_magenta'].copy()
CMAP_TF.set_bad(P.GREY_BG)       # a TF that could not be tested draws grey, never as 'zero'
CB_MATCHED = f'eRegulon enrichment\n(\u0394AUC vs matched null)\n* FDR < {FDR}'


def tf_heatmap(Dm, Qm, labels, width, path, suptitle, cbar_label=CB_MATCHED, cell_mm=6.0):
    """TF x factor AUC difference on the shared scale, * where FDR < FDR."""
    M = Dm.rename(columns=PUB_FACTORS)
    fig = P.niche_factor_heatmap(M, niche_order=list(M.columns), labels=labels, cmap=CMAP_TF,
                                 vlim=VLIM_TF, width=width, cell_mm=cell_mm, annot_n=False,
                                 cbar_label=cbar_label, suptitle=suptitle)
    ax, cax = fig.axes[0], fig.axes[1]
    # the helper prints the colour-bar ends to 1 dp, which rounds an AUC range away
    cax.set_yticklabels([f'{-VLIM_TF:.2f}', '0', f'{VLIM_TF:.2f}'])
    for i, tf in enumerate(Dm.index):
        for j, f in enumerate(Dm.columns):
            if Qm.loc[tf, f] < FDR:          # NaN compares False -- untested cells stay bare
                dark = abs(Dm.loc[tf, f]) > 0.55 * VLIM_TF
                # an asterisk sits high in its em box; nudge it down to the cell centre
                ax.text(j, i + 0.25, '*', ha='center', va='center', fontsize=7,
                        color='white' if dark else 'black')
    P.save(fig, path)
    plt.show()


KEY_LABELS = {tf: tf if tf not in not_scored else f'{tf} (n/t)' for tf in KEY_TFS}

# Fig 8a is drawn grouped in the next cell

# --- Fig 8b: the original key TFs across all 16 factors (sup) -- is the signal specific?
tf_heatmap(D_key, Q_key, KEY_LABELS, 'one-half',
           f'{FIGS}/Fig8b_scenicplusV2_MWU_keyTFs_allFactors_{today}.pdf',
           'SCENIC+ eRegulons — all 16 factors')

# --- Fig 8c: NEW curated candidates x the five selected factors (sup)
if CAND:
    tf_heatmap(D_cand, Q_cand, {tf: f'{tf} [{cls_of(tf)}]' for tf in CAND}, 'single',
               f'{FIGS}/Fig8c_scenicplusV2_MWU_candidateTFs_selectedFactors_{today}.pdf',
               'Other TFs in the selected factors', cell_mm=min(6.0, 150.0 / len(CAND)))
    print('row tags: I inflammatory, A adipogenic, F profibrotic')

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig8b_scenicplusV2_MWU_keyTFs_allFactors_2026-10-28.{pdf}   130 x 117 mm
-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig8c_scenicplusV2_MWU_candidateTFs_selectedFactors_2026-10-28.{pdf}   88 x 67 mm
row tags: I inflammatory, A adipogenic, F profibrotic


### Fig 8a — key TFs grouped by programme (v3 revision)

VK's revision (2026-10-28) of the main panel. It shows the same quantity as before: `dAUC` against
the matched null, with `*` for FDR < 0.05, BH across all tests. Three things change:

- **TFs in four groups**, in this order:
  - AP-1 / inflammatory: FOSL1, MYC, GATAD2A, CEBPB
  - profibrotic / activated: TEAD4, HIF1A, ZEB2, TWIST1, CREB3L1, NFIX
  - steroid: AR, NR3C1
  - adipogenic: PPARG, BCL6, EBF2, CEBPA

  TEAD4, HIF1A, ZEB2 and CEBPA come from the §10.1 screen. NR3C1 was not on the original key list.
  ATF6 and, for now, ATF4 are left out of this panel; both stay in `Fig8b` and `Fig8d`.
- **Factors grouped as in §8**, with S12 added: inflammatory S7 / S12, fibrotic S1 / S10, then
  adipogenic S11 / S15.
- **Red-blue colour scale**, centred on 0 and scaled to this panel alone. It is not on the shared
  scale of `8b`–`8d`, which keep their earlier colours and file names.

The file is named `Fig8a_…_keyTFs_groupedSteroid_RdBu`, so the earlier versions of `Fig8a` are
kept: the ungrouped one and the three-group `_grouped_RdBu`.

In [44]:
# VK 2026-10-28: TFs grouped by programme, in this order; factors grouped as in §8
TF_GROUPS = {'AP-1 / inflammatory':     ['FOSL1', 'MYC', 'GATAD2A', 'CEBPB'],
             'Profibrotic / activated': ['TEAD4', 'HIF1A', 'ZEB2', 'TWIST1', 'CREB3L1', 'NFIX'],
             'Steroid':                 ['AR', 'NR3C1'],
             'Adipogenic':              ['PPARG', 'BCL6', 'EBF2', 'CEBPA']}
GROUPED_TFS = [tf for tfs in TF_GROUPS.values() for tf in tfs]

D_grp, Q_grp, PCT_grp = (tf_matrix(GROUPED_TFS, v, SELECTED)
                         for v in ('dAUC', 'padj', 'pct_dAUC'))
nt_grp = D_grp.index[D_grp.isna().all(axis=1)].tolist()
print('TFs not tested (no eRegulon, or < TMIN targets in the basis):', nt_grp or 'none')
pd.concat({'dAUC': D_grp, 'padj': Q_grp, 'pct_dAUC': PCT_grp}, axis=1).to_csv(
    f'{RES}/Fig8a_scenicplusV2_MWU_keyTFs_groupedSteroid_{today}.csv')

# the panel's own scale, rounded up to 0.05 so the ticks are round numbers
VLIM_GRP = float(np.ceil(np.nanmax(np.abs(D_grp.values)) * 20) / 20)
fig = P.grouped_heatmap(
    D_grp, row_groups=TF_GROUPS, col_groups=SELECTED_GROUPS, col_labels=PUB_FACTORS,
    labels={tf: f'{tf} (n/t)' for tf in nt_grp}, marks=Q_grp < FDR, cmap=CMAP_RB,
    vlim=VLIM_GRP, cbar_ticks=[-VLIM_GRP, -VLIM_GRP / 2, 0, VLIM_GRP / 2, VLIM_GRP],
    cbar_fmt='{:.2f}', cbar_label=CB_MATCHED, width='single', cell_mm=5.0,
    suptitle='SCENIC+ eRegulons in factor loadings')
P.save(fig, f'{FIGS}/Fig8a_scenicplusV2_MWU_keyTFs_groupedSteroid_RdBu_{today}.pdf')
plt.show()

star = (Q_grp < FDR).map(lambda b: '*' if b else '')
print(f'dAUC vs matched null (* FDR < {FDR}, BH across all tests)')
print((D_grp.map(lambda d: f'{d:+.2f}') + star).to_string())

TFs not tested (no eRegulon, or < TMIN targets in the basis): none


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig8a_scenicplusV2_MWU_keyTFs_groupedSteroid_RdBu_2026-10-28.{pdf}   88 x 123 mm
dAUC vs matched null (* FDR < 0.05, BH across all tests)
factor       S7     S12      S1     S10     S11     S15
tf                                                     
FOSL1    +0.16*  +0.23*  +0.24*   +0.01   -0.16   -0.04
MYC      +0.09*  +0.14*   +0.03   -0.05   -0.03   -0.00
GATAD2A  +0.16*  +0.18*  +0.37*   +0.02   -0.11   -0.03
CEBPB    +0.14*   +0.06   +0.02   -0.02   +0.02   +0.02
TEAD4    +0.13*  +0.11*  +0.26*  +0.10*   -0.16   -0.03
HIF1A    +0.12*   +0.05  +0.17*  +0.10*   -0.07   -0.02
ZEB2      +0.09   +0.02  +0.16*   -0.03   -0.07   +0.00
TWIST1    -0.01   -0.10   -0.03  +0.12*  +0.12*  +0.20*
CREB3L1   -0.05   -0.01  +0.28*  +0.36*   -0.03   -0.02
NFIX      -0.04   -0.01  +0.23*  +0.24*   -0.06   -0.04
AR        -0.00   -0.16   -0.08   -0.02  +0.26*   -0.04
NR3C1     -0.09   -0.10   -0.03  

### 10.2 Diagnostic — the same test on the raw loadings

The recipe VK described, unchanged: a one-sided Mann–Whitney of each factor's **raw** loading row
(targets vs the rest, no cutoff), AUC = U / (n₁ n₂), BH across all tests, no permutation. It sits
beside the matched version to answer two questions empirically:

- **does a raw-loadings call follow the targets' abundance?** Per factor: `rho(raw W, gene
  abundance)` says how abundance-tracking that factor's loadings are; `rho(AUC raw, target
  abundance)` and `rho(dAUC, target abundance)` say how strongly each method's eRegulon ranking
  follows the mean abundance of the eRegulon's targets.
- **how specific is each method?** An eRegulon "significant" in most of the 16 factors is
  describing its target genes, not any one factor — so the number of factors each eRegulon is
  significant in is the specificity read-out.

`Fig8d` is the original key-TF panel under this recipe, on the same colour scale as `Fig8b`.

In [45]:
# per eRegulon: mean abundance of its in-basis targets relative to the basis mean
log2_ab = pd.Series(np.log2((Mem.T @ abundance / n1) / abundance.mean()), index=sources)
res_sc['log2_target_abundance'] = res_sc['source'].map(log2_ab)

diag = []
for f in Zg.index:
    d = res_sc[res_sc['factor'] == f]
    diag.append({'factor': f,
                 'rho(raw W, gene abundance)': spearmanr(W.loc[f, universe], abundance)[0],
                 'rho(AUC raw, target abundance)': spearmanr(d['auc_raw'], d['log2_target_abundance'])[0],
                 'rho(dAUC, target abundance)': spearmanr(d['dAUC'], d['log2_target_abundance'])[0],
                 'rho(AUC raw, dAUC)': spearmanr(d['auc_raw'], d['dAUC'])[0],
                 'FDR hits, raw': int((d['padj_raw'] < FDR).sum()),
                 'FDR hits, matched': int((d['padj'] < FDR).sum())})
diag = pd.DataFrame(diag).set_index('factor')
diag.to_csv(f'{RES}/Fig8d_scenicplusV2_MWU_rawVsMatched_diagnostic_{today}.csv')
res_sc.to_csv(f'{RES}/Fig8_scenicplusV2_MWU_factorEnrichment_allFactors_{today}.csv', index=False)
print(diag.round(2).to_string())

# specificity: in how many of the 16 factors is each eRegulon significant?
nsig = pd.DataFrame({
    'raw loadings': res_sc[res_sc['padj_raw'] < FDR].groupby('source').size(),
    'matched null': res_sc[res_sc['padj'] < FDR].groupby('source').size()}
    ).reindex(sources).fillna(0).astype(int)
print(f'\nfactors (of 16) each eRegulon is significant in -- {len(sources)} eRegulons')
print(nsig.describe().loc[['mean', '50%', 'max']].round(1).to_string())
print('eRegulons significant in >= 8 of 16 factors:', (nsig >= 8).sum().to_dict())

print('\nkey TFs x selected factors: raw-loadings AUC - 0.5 | matched dAUC')
print(pd.concat({'raw AUC-0.5': RAW_key, 'matched dAUC': D_key[TF_FACTORS]}, axis=1)
      .round(3).to_string())

        rho(raw W, gene abundance)  rho(AUC raw, target abundance)  rho(dAUC, target abundance)  rho(AUC raw, dAUC)  FDR hits, raw  FDR hits, matched
factor                                                                                                                                               
S1                            0.24                            0.40                         0.10                0.74             37                 23
S2                           -0.15                           -0.69                        -0.59                0.93             74                 62
S3                            0.49                            0.78                        -0.22                0.09             57                 11
S4                            0.13                           -0.12                        -0.22                0.44             60                 31
S5                            0.47                            0.73                        -0.27     

In [46]:
# --- Fig 8d: the key-TF panel under the raw-loadings recipe, same scale as 8b
tf_heatmap(RAW_key, QRAW_key, KEY_LABELS, 'single',
           f'{FIGS}/Fig8d_scenicplusV2_MWU_keyTFs_rawLoadings_diagnostic_{today}.pdf',
           'Raw-loadings recipe (diagnostic)',
           cbar_label=f'AUC \u2212 0.5, raw loadings\n(no null)\n* FDR < {FDR}')

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/Fb_ageing_publication/Fig8d_scenicplusV2_MWU_keyTFs_rawLoadings_diagnostic_2026-10-28.{pdf}   88 x 115 mm


In [47]:
# Cross-check against CollecTRI (nb09). nb09's z came from SD.null_for, whose permutation is not
# abundance-matched (see the scoring cell), so read it as a near-raw ULM score on the same per-gene
# statistic. The two networks are also built very differently (literature curation vs multiome
# inference): agreement is not expected to be perfect, but a TF strongly positive in one and
# strongly negative in the other deserves a look before it goes in a figure.
ct = pd.read_csv(COLLECTRI_RES)
Z_ct = (ct[ct['source'].isin(KEY_TFS)].pivot(index='source', columns='factor', values='z')
        .reindex(index=KEY_TFS, columns=TF_FACTORS))
pair = pd.concat({'SCENIC+ dAUC': D_key[TF_FACTORS].stack(), 'CollecTRI z': Z_ct.stack()},
                 axis=1).dropna()
rho, p = spearmanr(pair['SCENIC+ dAUC'], pair['CollecTRI z'])
print('key TFs CollecTRI could score:', Z_ct.dropna(how='all').index.tolist())
print(f'TF x factor pairs scored by both: {len(pair)} | Spearman rho {rho:.2f} (p = {p:.1e})')
pd.concat({'SCENIC+ v2 dAUC': D_key[TF_FACTORS].round(3), 'CollecTRI z': Z_ct.round(1)}, axis=1)

key TFs CollecTRI could score: ['PPARG', 'BCL6', 'TWIST1', 'AR', 'CEBPB', 'MYC', 'FOSL1', 'ATF4', 'ATF6']
TF x factor pairs scored by both: 45 | Spearman rho 0.26 (p = 9.0e-02)


SCENIC+ v2 dAUC                             CollecTRI z            \
factor               S1     S7    S10    S11    S15          S1   S7  S10   
PPARG            -0.028  0.016 -0.016  0.181  0.250        -1.3  0.5 -2.2   
BCL6             -0.149 -0.052 -0.076  0.265  0.246        -0.5  1.9  0.7   
TWIST1           -0.033 -0.006  0.122  0.123  0.196         0.4  3.1  1.0   
AR               -0.081 -0.003 -0.020  0.262 -0.040        -0.5  4.3  1.7   
EBF2             -0.137 -0.032 -0.013  0.160  0.129         NaN  NaN  NaN   
CEBPB             0.020  0.139 -0.015  0.019  0.018        -0.6  5.4  0.0   
MYC               0.030  0.090 -0.050 -0.034 -0.004         0.8  6.7 -0.6   
FOSL1             0.236  0.157  0.012 -0.163 -0.040         0.3 -0.3  1.4   
GATAD2A           0.374  0.156  0.015 -0.105 -0.030         NaN  NaN  NaN   
CREB3L1           0.283 -0.051  0.365 -0.033 -0.024         NaN  NaN  NaN   
NFIX              0.230 -0.040  0.241 -0.057 -0.038         NaN  NaN  NaN   
ATF4              0.134  0.058  0.239 -0.087  0.051         1.0  2.6 -0.8   
ATF6             -0.088 -0.037 -0.147 -0.090  0.156         0.1  2.3  1.9   

                   
factor   S11  S15  
PPARG   -0.8  8.2  
BCL6     1.7 -0.6  
TWIST1   1.4  0.4  
AR       2.2 -1.4  
EBF2     NaN  NaN  
CEBPB    0.3  4.4  
MYC     -0.4 -0.6  
FOSL1   -0.6 -1.2  
GATAD2A  NaN  NaN  
CREB3L1  NaN  NaN  
NFIX     NaN  NaN  
ATF4    -1.4 -1.8  
ATF6    -2.0 -1.5

## 11. Files written

In [48]:
import glob
print('FIGURES')
for f in sorted(glob.glob(f'{FIGS}/*{today}*')):
    print(f'  {os.path.getsize(f) / 1e6:6.2f} MB  {os.path.basename(f)}')
print('\nTABLES')
for f in sorted(glob.glob(f'{RES}/*{today}*')) + sorted(glob.glob(f'{RES}/Fig4_freq_*_kruskal.csv')):
    print(f'  {os.path.getsize(f) / 1e3:6.1f} kB  {os.path.basename(f)}')

FIGURES
    8.50 MB  Fig10a_factorProjection_umap_cmx_smoothed_RdBu_2026-10-28.pdf
    9.60 MB  Fig10b_factorProjection_umap_cmx_byModality_RdBu_2026-10-28.pdf
    8.07 MB  Fig10c_factorUsage_umap_cmx_RNAonly_smoothed_RdBu_2026-10-28.pdf
    9.39 MB  Fig10d_factorProjection_umap_cmx_unsmoothed_RdBu_2026-10-28.pdf
    2.34 MB  Fig1a2_umap_donor_a_2026-10-28.pdf
    3.52 MB  Fig1a_umap_a_2026-10-28.pdf
    2.25 MB  Fig1b2_umap_donor_b_2026-10-28.pdf
    3.64 MB  Fig1b_umap_b_2026-10-28.pdf
    0.06 MB  Fig2a_dotplot_activity_scellMultiome_2026-10-28.pdf
    0.06 MB  Fig2a_dotplot_activity_scellMultiome_unscaled_2026-10-28.pdf
    0.05 MB  Fig2b_dotplot_identity_scellMultiome_2026-10-28.pdf
    0.05 MB  Fig2b_dotplot_identity_scellMultiome_unscaled_2026-10-28.pdf
    0.06 MB  Fig3a_dotplot_adiposeDifferentiation_cmx_2026-10-28.pdf
    0.06 MB  Fig3a_dotplot_adiposeDifferentiation_cmx_unscaled_2026-10-28.pdf
    0.06 MB  Fig3b_dotplot_identityActivity_cmx_2026-10-28.pdf
    0.06 MB  Fig3b_